In [ ]:
"""
ABC Foodmart - Checkpoint 4 Data Pipeline
APAN 5310: SQL & Relational Databases

Purpose
-------
1. Generate reproducible synthetic operational data for the 19 tables in the
   Checkpoint 3 PostgreSQL schema.
2. Transform and validate the generated CSV staging files.
3. Load the transformed data into the abc_foodmart schema in dependency order.

Important
---------
- The generated records are fictional test data.
- The abc_foodmart schema must already exist before loading. For a clean build,
  use checkpoint3_schema_candidate.sql from this candidate folder.
- The schema file already contains sample rows. By default, this script removes
  those rows before loading the new synthetic dataset.
- The database triggers remain active. Sales-item inserts automatically reduce
  inventory, so this pipeline does not separately create "sale" adjustments.
  
Env Requirements
---------
- pandas
- psycopg2-binary

"""

In [ ]:
from __future__ import annotations

import argparse
import csv
import json
import getpass
import logging
import math
import os
import random
import re
import sys
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import date, datetime, time, timedelta
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path
from typing import Any, Iterable, Sequence

try:
    import pandas as pd
except ImportError as exc:
    raise SystemExit(
        "pandas is required. Install it with: pip install pandas"
    ) from exc

try:
    import psycopg2
    from psycopg2 import sql
    from psycopg2.extras import RealDictCursor, execute_values
except ImportError:
    # CSV generation and validation do not require a database driver. Delay the
    # dependency error until a PostgreSQL load is actually requested.
    psycopg2 = None
    sql = None
    execute_values = None
    

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

SCHEMA = "abc_foodmart"
RANDOM_SEED = 5310
MONEY = Decimal("0.01")
TAX_RATE = Decimal("0.08875")

DEFAULT_START_DATE = date(2025, 11, 1)
DEFAULT_END_DATE = date(2026, 7, 31)
DEFAULT_SALES_COUNT = 36_000
DEFAULT_EMPLOYEE_COUNT = 80

# The eight protected reference/master CSVs must stay byte-identical to the
# approved Option A candidate.  Their original random stream included the
# legacy 19-month staffing generation before products and vendor products were
# created, so the protected-data builder intentionally replays that harmless
# reference-only stream.  Every Option B fact area then starts from its own
# fixed seed; changing sales or staffing logic cannot drift another area.
PROTECTED_REFERENCE_START_DATE = date(2025, 1, 1)
DOMAIN_SEEDS = {
    "staffing": 531_001,
    "sales": 531_002,
    "purchasing": 531_003,
    "inventory_loss": 531_004,
    "expenses": 531_005,
}

OPENING_INVENTORY_DAYS = 10
REPLENISHMENT_DAYS = 7
PURCHASE_SAFETY_PERCENT = 5
REORDER_SAFETY_UNITS = 35

# Dataset-lifecycle rules. A submitted order may remain genuinely open when it
# is not yet due or is no more than one week overdue at the snapshot cutoff.
# These constants affect deterministic post-processing only; they do not add
# any calls to the module-level random generator.
OPEN_ORDER_GRACE_DAYS = 7
MAX_COMPLETED_SHIFT_HOURS = 8
MIN_COMPLETED_SHIFT_HOURS = 4

TABLE_ORDER = [
    "stores",
    "job_roles",
    "employees",
    "staff_shifts",
    "time_off_requests",
    "product_categories",
    "products",
    "store_inventory",
    "inventory_adjustments",
    "vendors",
    "vendor_products",
    "purchase_orders",
    "purchase_order_items",
    "deliveries",
    "sales_transactions",
    "sales_items",
    "customer_returns",
    "payments",
    "operating_expenses",
]

IDENTITY_COLUMNS = {
    "stores": "store_id",
    "job_roles": "role_id",
    "employees": "employee_id",
    "staff_shifts": "shift_id",
    "time_off_requests": "request_id",
    "product_categories": "category_id",
    "products": "product_id",
    "inventory_adjustments": "adjustment_id",
    "vendors": "vendor_id",
    "vendor_products": "vendor_product_id",
    "purchase_orders": "purchase_order_id",
    "purchase_order_items": "purchase_order_item_id",
    "deliveries": "delivery_id",
    "sales_transactions": "sale_id",
    "sales_items": "sale_item_id",
    "customer_returns": "return_id",
    "payments": "payment_id",
    "operating_expenses": "expense_id",
}

# Tables must be truncated in reverse dependency order.
TRUNCATE_TABLES = list(reversed(TABLE_ORDER))

FIRST_NAMES = [
    "Emily", "Jason", "Maria", "Daniel", "Aisha", "Kevin", "Sophia", "Michael",
    "Olivia", "Ethan", "Mia", "Noah", "Chloe", "Lucas", "Grace", "Henry",
    "Isabella", "Liam", "Zoe", "David", "Nina", "Ryan", "Sara", "Owen",
]
LAST_NAMES = [
    "Chen", "Lee", "Garcia", "Kim", "Patel", "Wong", "Brown", "Wilson",
    "Martinez", "Davis", "Lopez", "Clark", "Lewis", "Walker", "Hall", "Young",
    "King", "Scott", "Green", "Baker", "Adams", "Nelson", "Carter", "Mitchell",
]

PRODUCT_LIBRARY = {
    "Beverages": [
        ("Bottled Water", "PureSpring", "24 pack", "6.99"),
        ("Orange Juice", "SunValley", "52 oz", "4.49"),
        ("Sparkling Water", "ClearWave", "12 pack", "5.99"),
        ("Cola", "MetroFizz", "2 liter", "2.49"),
        ("Ground Coffee", "Morning Peak", "12 oz", "8.99"),
        ("Green Tea", "Eastern Leaf", "20 bags", "4.79"),
    ],
    "Snacks": [
        ("Potato Chips", "CrispyCo", "8 oz", "3.29"),
        ("Pretzels", "SnackHouse", "12 oz", "3.49"),
        ("Trail Mix", "NatureFuel", "10 oz", "6.49"),
        ("Chocolate Cookies", "SweetStreet", "14 oz", "4.29"),
        ("Granola Bars", "DailyGrain", "6 pack", "4.99"),
        ("Popcorn", "MovieNight", "6 bags", "5.29"),
    ],
    "Fresh Produce": [
        ("Bananas", "FreshFarm", "1 lb", "0.79"),
        ("Apples", "Hudson Orchard", "3 lb bag", "5.49"),
        ("Tomatoes", "FreshFarm", "1 lb", "2.49"),
        ("Avocados", "Green Valley", "4 count", "4.99"),
        ("Spinach", "Garden Fresh", "8 oz", "3.49"),
        ("Carrots", "Garden Fresh", "2 lb bag", "2.99"),
    ],
    "Dairy": [
        ("Whole Milk", "DairyBest", "1 gallon", "4.99"),
        ("Greek Yogurt", "DairyBest", "32 oz", "6.49"),
        ("Cheddar Cheese", "FarmTable", "8 oz", "4.79"),
        ("Butter", "FarmTable", "16 oz", "5.29"),
        ("Eggs", "Sunny Acres", "12 count", "4.69"),
        ("Oat Milk", "PlantDay", "64 oz", "4.99"),
    ],
    "Household": [
        ("Paper Towels", "CleanHome", "6 rolls", "8.99"),
        ("Dish Soap", "CleanHome", "24 oz", "3.99"),
        ("Laundry Detergent", "BrightWash", "64 oz", "12.99"),
        ("Trash Bags", "StrongHold", "30 count", "9.49"),
        ("All Purpose Cleaner", "CleanHome", "32 oz", "4.49"),
        ("Toilet Paper", "SoftHome", "12 rolls", "11.99"),
    ],
    "Bakery": [
        ("Whole Wheat Bread", "DailyBake", "20 oz", "3.99"),
        ("Bagels", "DailyBake", "6 count", "4.49"),
        ("Croissants", "DailyBake", "4 count", "5.99"),
        ("Blueberry Muffins", "DailyBake", "4 count", "5.49"),
        ("Tortillas", "CasaFresh", "10 count", "3.79"),
        ("Dinner Rolls", "DailyBake", "12 count", "4.29"),
    ],
    "Frozen Food": [
        ("Frozen Pizza", "QuickMeal", "12 inch", "7.99"),
        ("Frozen Vegetables", "Green Valley", "16 oz", "3.49"),
        ("Ice Cream", "SweetCream", "1.5 quart", "6.99"),
        ("Chicken Nuggets", "QuickMeal", "24 oz", "8.49"),
        ("Frozen Waffles", "MorningTable", "10 count", "4.99"),
        ("French Fries", "QuickMeal", "32 oz", "5.49"),
    ],
    "Personal Care": [
        ("Shampoo", "DailyCare", "16 oz", "6.99"),
        ("Toothpaste", "BrightSmile", "6 oz", "4.49"),
        ("Hand Soap", "DailyCare", "12 oz", "3.49"),
        ("Body Wash", "DailyCare", "18 oz", "7.49"),
        ("Deodorant", "FreshDay", "2.6 oz", "5.99"),
        ("Facial Tissues", "SoftHome", "3 boxes", "6.49"),
    ],
}

VENDOR_NAMES = [
    "Metro Grocery Supply",
    "Fresh Farm Distributors",
    "Household Wholesale NY",
    "Empire Beverage Partners",
    "Queens Dairy Cooperative",
    "Brooklyn Bakery Supply",
    "Atlantic Frozen Foods",
    "Daily Essentials Wholesale",
    "Hudson Valley Produce",
    "Tri-State Consumer Goods",
]

EMPTY_TABLE_COLUMNS = {
    "customer_returns": [
        "return_id",
        "sale_item_id",
        "processed_by_employee_id",
        "return_time",
        "quantity_returned",
        "refund_amount",
        "return_reason",
    ],
}


@dataclass(frozen=True)
class PipelineConfig:
    output_dir: Path
    start_date: date
    end_date: date
    sales_count: int
    employee_count: int
    reset_before_load: bool
    generate_only: bool
    load_only: bool


# ---------------------------------------------------------------------------
# Utility functions
# ---------------------------------------------------------------------------

def money(value: Any) -> Decimal:
    """Convert a value to a two-decimal Decimal."""
    return Decimal(str(value)).quantize(MONEY, rounding=ROUND_HALF_UP)


def seed_fact_domain(domain: str) -> None:
    """Start one isolated deterministic fact-generation random stream."""
    random.seed(DOMAIN_SEEDS[domain])


def random_date(start: date, end: date) -> date:
    if end < start:
        raise ValueError("End date cannot be earlier than start date.")
    return start + timedelta(days=random.randint(0, (end - start).days))


def random_datetime(start: date, end: date, start_hour: int = 8, end_hour: int = 21) -> datetime:
    selected_date = random_date(start, end)
    hour = random.randint(start_hour, max(start_hour, end_hour - 1))
    minute = random.choice([0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55])
    return datetime.combine(selected_date, time(hour, minute))


def standardize_column_name(name: str) -> str:
    name = name.strip()
    name = re.sub(r"[^A-Za-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_").lower()


def normalize_text(value: Any) -> Any:
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    if text.lower() in {"", "n/a", "na", "null", "none", "unknown", "-"}:
        return None
    return re.sub(r"\s+", " ", text)


def safe_csv_value(value: Any) -> Any:
    if isinstance(value, Decimal):
        return format(value, "f")
    if isinstance(value, (datetime, date, time)):
        return value.isoformat(sep=" ") if isinstance(value, datetime) else value.isoformat()
    return value


def write_csv(path: Path, rows: Sequence[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    columns = (
        list(rows[0].keys())
        if rows
        else EMPTY_TABLE_COLUMNS.get(path.stem)
    )
    if columns is None:
        raise ValueError(f"No rows or column definition were generated for {path.stem}.")
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=columns)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: safe_csv_value(row.get(key)) for key in columns})


def dataframe_to_records(df: pd.DataFrame) -> list[tuple[Any, ...]]:
    records = []
    for row in df.itertuples(index=False, name=None):
        cleaned = []
        for value in row:
            if pd.isna(value):
                cleaned.append(None)
            elif isinstance(value, pd.Timestamp):
                cleaned.append(value.to_pydatetime())
            else:
                cleaned.append(value)
        records.append(tuple(cleaned))
    return records


# ---------------------------------------------------------------------------
# Synthetic data generation
# ---------------------------------------------------------------------------

def generate_stores() -> list[dict[str, Any]]:
    return [
        {
            "store_id": 1,
            "store_name": "ABC Foodmart - Flushing",
            "borough": "Queens",
            "street_address": "136-20 Roosevelt Ave",
            "city": "New York",
            "state": "NY",
            "zip_code": "11354",
            "opening_time": "08:00:00",
            "closing_time": "22:00:00",
            "operational_status": "active",
        },
        {
            "store_id": 2,
            "store_name": "ABC Foodmart - Jamaica",
            "borough": "Queens",
            "street_address": "90-15 Parsons Blvd",
            "city": "New York",
            "state": "NY",
            "zip_code": "11432",
            "opening_time": "08:00:00",
            "closing_time": "21:00:00",
            "operational_status": "active",
        },
        {
            "store_id": 3,
            "store_name": "ABC Foodmart - Brooklyn Heights",
            "borough": "Brooklyn",
            "street_address": "120 Montague St",
            "city": "New York",
            "state": "NY",
            "zip_code": "11201",
            "opening_time": "09:00:00",
            "closing_time": "21:00:00",
            "operational_status": "active",
        },
        {
            "store_id": 4,
            "store_name": "ABC Foodmart - Williamsburg",
            "borough": "Brooklyn",
            "street_address": "188 Bedford Ave",
            "city": "New York",
            "state": "NY",
            "zip_code": "11211",
            "opening_time": "09:00:00",
            "closing_time": "22:00:00",
            "operational_status": "planned",
        },
        {
            "store_id": 5,
            "store_name": "ABC Foodmart - Park Slope",
            "borough": "Brooklyn",
            "street_address": "310 7th Ave",
            "city": "New York",
            "state": "NY",
            "zip_code": "11215",
            "opening_time": "09:00:00",
            "closing_time": "21:00:00",
            "operational_status": "planned",
        },
    ]


def generate_job_roles() -> list[dict[str, Any]]:
    roles = [
        ("Store Manager", "Manages store operations and staffing"),
        ("Assistant Manager", "Supports the manager and daily operations"),
        ("Cashier", "Handles checkout and customer payments"),
        ("Stock Clerk", "Receives deliveries and maintains shelves"),
        ("Inventory Specialist", "Monitors inventory and reorder needs"),
        ("Customer Service Associate", "Supports customers and processes returns"),
    ]
    return [
        {"role_id": idx, "role_name": role, "role_description": description}
        for idx, (role, description) in enumerate(roles, start=1)
    ]


def generate_employees(employee_count: int, end_date: date) -> list[dict[str, Any]]:
    active_store_ids = [1, 2, 3]
    role_ids = [1, 2, 3, 4, 5, 6]
    rows: list[dict[str, Any]] = []

    # Ensure one manager per active store.
    for store_id in active_store_ids:
        idx = len(rows) + 1
        first = FIRST_NAMES[(idx - 1) % len(FIRST_NAMES)]
        last = LAST_NAMES[(idx - 1) % len(LAST_NAMES)]
        rows.append(
            {
                "employee_id": idx,
                "store_id": store_id,
                "role_id": 1,
                "first_name": first,
                "last_name": last,
                "email": f"{first.lower()}.{last.lower()}{idx}@abcfoodmart.com",
                "hire_date": random_date(date(2021, 1, 1), min(end_date, date(2025, 12, 31))),
                "hourly_pay_rate": money(random.uniform(30, 38)),
                "employment_status": "active",
            }
        )

    while len(rows) < employee_count:
        idx = len(rows) + 1
        first = random.choice(FIRST_NAMES)
        last = random.choice(LAST_NAMES)
        store_id = random.choice(active_store_ids)
        role_id = random.choices(role_ids[1:], weights=[10, 35, 28, 12, 15], k=1)[0]
        pay_ranges = {
            2: (24, 30),
            3: (17, 22),
            4: (18, 23),
            5: (21, 27),
            6: (18, 23),
        }
        low, high = pay_ranges[role_id]
        status = random.choices(
            ["active", "on_leave", "terminated"], weights=[88, 5, 7], k=1
        )[0]
        rows.append(
            {
                "employee_id": idx,
                "store_id": store_id,
                "role_id": role_id,
                "first_name": first,
                "last_name": last,
                "email": f"{first.lower()}.{last.lower()}{idx}@abcfoodmart.com",
                "hire_date": random_date(date(2021, 1, 1), end_date),
                "hourly_pay_rate": money(random.uniform(low, high)),
                "employment_status": status,
            }
        )
    return rows


def generate_staff_shifts(
    employees: list[dict[str, Any]],
    start_date: date,
    end_date: date,
) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    shift_id = 1
    for employee in employees:
        if employee["employment_status"] == "terminated":
            continue
        # Roughly 2-4 shifts per employee per month across the simulation period.
        months = max(1, ((end_date.year - start_date.year) * 12 + end_date.month - start_date.month + 1))
        shift_count = random.randint(2 * months, 4 * months)
        for _ in range(shift_count):
            day = random_date(max(start_date, employee["hire_date"]), end_date)
            start_hour = random.choice([8, 9, 10, 11, 12, 13, 14])
            scheduled_start = datetime.combine(day, time(start_hour, 0))
            duration = random.choice([4, 6, 8])
            scheduled_end = scheduled_start + timedelta(hours=duration)
            status = random.choices(
                ["completed", "scheduled", "cancelled", "missed"],
                weights=[82, 8, 6, 4],
                k=1,
            )[0]

            actual_start = None
            actual_end = None
            if status == "completed":
                actual_start = scheduled_start + timedelta(minutes=random.randint(-10, 20))
                actual_end = scheduled_end + timedelta(minutes=random.randint(-10, 20))

            rows.append(
                {
                    "shift_id": shift_id,
                    "employee_id": employee["employee_id"],
                    "store_id": employee["store_id"],
                    "scheduled_start": scheduled_start,
                    "scheduled_end": scheduled_end,
                    "actual_start": actual_start,
                    "actual_end": actual_end,
                    "shift_status": status,
                }
            )
            shift_id += 1
    return rows


def generate_time_off_requests(
    employees: list[dict[str, Any]],
    start_date: date,
    end_date: date,
) -> list[dict[str, Any]]:
    reasons = [
        "Personal appointment", "Family responsibility", "Vacation",
        "Medical appointment", "Personal day", "School commitment",
    ]
    rows = []
    request_id = 1
    candidates = [e for e in employees if e["employment_status"] != "terminated"]
    # Avoid manufacturing twenty leave requests inside a one- or two-day
    # custom test window. The default 19-month run remains unchanged at one
    # request per eligible employee.
    window_days = (end_date - start_date).days + 1
    request_count = max(len(candidates), min(20, window_days))
    for _ in range(request_count):
        employee = random.choice(candidates)
        request_date = random_date(start_date, end_date)
        # Preserve the deterministic random sequence while correcting any
        # impossible pre-hire request date.
        request_date = max(request_date, employee["hire_date"])
        start = min(end_date, request_date + timedelta(days=random.randint(3, 45)))
        end = min(end_date, start + timedelta(days=random.randint(0, 5)))
        status = random.choices(
            ["approved", "denied", "pending", "cancelled"],
            weights=[55, 18, 17, 10],
            k=1,
        )[0]
        review_date = None
        if status in {"approved", "denied"}:
            review_date = min(start, request_date + timedelta(days=random.randint(0, 5)))
        rows.append(
            {
                "request_id": request_id,
                "employee_id": employee["employee_id"],
                "request_date": request_date,
                "start_date": start,
                "end_date": end,
                "reason": random.choice(reasons),
                "request_status": status,
                "review_date": review_date,
            }
        )
        request_id += 1
    return rows


def reconcile_time_off_lifecycle(
    time_off_requests: list[dict[str, Any]],
    dataset_cutoff: date,
) -> tuple[int, int]:
    """Resolve historical pending requests without consuming random values."""
    resolved_as_denied = 0
    resolved_as_cancelled = 0
    for request in time_off_requests:
        if (
            request["request_status"] != "pending"
            or request["end_date"] >= dataset_cutoff
        ):
            continue

        # Use the stable request ID to retain a credible mix of employee-
        # cancelled and manager-denied historical requests. Requests that still
        # overlap the cutoff may remain pending.
        if request["request_id"] % 2 == 0:
            request["request_status"] = "denied"
            request["review_date"] = min(
                request["start_date"],
                request["request_date"] + timedelta(days=2),
            )
            resolved_as_denied += 1
        else:
            request["request_status"] = "cancelled"
            request["review_date"] = None
            resolved_as_cancelled += 1

    return resolved_as_denied, resolved_as_cancelled


def reconcile_staff_schedule(
    shifts: list[dict[str, Any]],
    time_off_requests: list[dict[str, Any]],
) -> tuple[int, int, int]:
    """Cancel impossible overlaps without changing IDs or random generation."""
    approved_ranges: dict[int, list[tuple[date, date]]] = defaultdict(list)
    for request in time_off_requests:
        if request["request_status"] == "approved":
            approved_ranges[request["employee_id"]].append(
                (request["start_date"], request["end_date"])
            )

    time_off_cancellations = 0
    for shift in shifts:
        if shift["shift_status"] == "cancelled":
            continue
        shift_date = shift["scheduled_start"].date()
        if any(
            start_date <= shift_date <= end_date
            for start_date, end_date in approved_ranges.get(
                shift["employee_id"], []
            )
        ):
            shift["shift_status"] = "cancelled"
            shift["actual_start"] = None
            shift["actual_end"] = None
            time_off_cancellations += 1

    shifts_by_employee: dict[int, list[dict[str, Any]]] = defaultdict(list)
    for shift in shifts:
        shifts_by_employee[shift["employee_id"]].append(shift)

    overlap_cancellations = 0
    for employee_shifts in shifts_by_employee.values():
        last_kept_end: datetime | None = None
        for shift in sorted(
            employee_shifts,
            key=lambda item: (
                item["scheduled_start"],
                item["scheduled_end"],
                item["shift_id"],
            ),
        ):
            if shift["shift_status"] == "cancelled":
                continue
            if (
                last_kept_end is not None
                and shift["scheduled_start"] < last_kept_end
            ):
                shift["shift_status"] = "cancelled"
                shift["actual_start"] = None
                shift["actual_end"] = None
                overlap_cancellations += 1
                continue
            last_kept_end = shift["scheduled_end"]

    actual_time_corrections = 0
    for employee_shifts in shifts_by_employee.values():
        previous_actual_end: datetime | None = None
        for shift in sorted(
            (
                item
                for item in employee_shifts
                if item["shift_status"] == "completed"
            ),
            key=lambda item: (item["actual_start"], item["shift_id"]),
        ):
            if (
                previous_actual_end is not None
                and shift["actual_start"] < previous_actual_end
            ):
                shift["actual_start"] = previous_actual_end
                actual_time_corrections += 1
            if shift["actual_end"] <= shift["actual_start"]:
                shift["shift_status"] = "cancelled"
                shift["actual_start"] = None
                shift["actual_end"] = None
                overlap_cancellations += 1
                continue
            previous_actual_end = shift["actual_end"]

    return time_off_cancellations, overlap_cancellations, actual_time_corrections


def _clamp_shift_interval_to_store_hours(
    interval_start: datetime,
    interval_end: datetime,
    opening_time: time,
    closing_time: time,
) -> tuple[datetime, datetime]:
    """Move an interval inside its same-day store hours without lengthening it."""
    opening = datetime.combine(interval_start.date(), opening_time)
    closing = datetime.combine(interval_start.date(), closing_time)
    duration = interval_end - interval_start
    if duration <= timedelta(0):
        raise ValueError("Shift interval must have a positive duration.")
    duration = min(duration, closing - opening)
    adjusted_start = max(opening, min(interval_start, closing - duration))
    return adjusted_start, adjusted_start + duration


def normalize_shifts_to_store_hours(
    shifts: list[dict[str, Any]],
    stores: list[dict[str, Any]],
) -> int:
    """Keep every scheduled and actual shift interval inside store hours."""
    store_hours = {
        store["store_id"]: (
            time.fromisoformat(str(store["opening_time"])),
            time.fromisoformat(str(store["closing_time"])),
        )
        for store in stores
    }
    corrected = 0
    for shift in shifts:
        opening_time, closing_time = store_hours[shift["store_id"]]
        scheduled_start, scheduled_end = _clamp_shift_interval_to_store_hours(
            shift["scheduled_start"],
            shift["scheduled_end"],
            opening_time,
            closing_time,
        )
        if (
            scheduled_start != shift["scheduled_start"]
            or scheduled_end != shift["scheduled_end"]
        ):
            corrected += 1
            shift["scheduled_start"] = scheduled_start
            shift["scheduled_end"] = scheduled_end

        if shift["shift_status"] == "completed":
            if shift["actual_start"] is None or shift["actual_end"] is None:
                raise ValueError(
                    f"Completed shift {shift['shift_id']} lacks actual times."
                )
            actual_start, actual_end = _clamp_shift_interval_to_store_hours(
                shift["actual_start"],
                shift["actual_end"],
                opening_time,
                closing_time,
            )
            if (
                actual_start != shift["actual_start"]
                or actual_end != shift["actual_end"]
            ):
                corrected += 1
                shift["actual_start"] = actual_start
                shift["actual_end"] = actual_end
        else:
            shift["actual_start"] = None
            shift["actual_end"] = None
    return corrected


def _intervals_overlap(
    first_start: datetime,
    first_end: datetime,
    second_start: datetime,
    second_end: datetime,
) -> bool:
    return first_start < second_end and second_start < first_end


def _coverage_interval(
    event_time: datetime,
    opening_time: time,
    closing_time: time,
) -> tuple[datetime, datetime]:
    """Create a deterministic 4-8 hour completed-shift window for an event."""
    opening = datetime.combine(event_time.date(), opening_time)
    closing = datetime.combine(event_time.date(), closing_time)
    rounded_start = event_time.replace(
        minute=(event_time.minute // 15) * 15,
        second=0,
        microsecond=0,
    )
    latest_start = closing - timedelta(hours=MIN_COMPLETED_SHIFT_HOURS)
    shift_start = max(opening, min(rounded_start, latest_start))
    shift_end = min(
        closing,
        shift_start + timedelta(hours=MAX_COMPLETED_SHIFT_HOURS),
    )
    return shift_start, shift_end


def reconcile_operational_staffing(
    shifts: list[dict[str, Any]],
    time_off_requests: list[dict[str, Any]],
    employees: list[dict[str, Any]],
    stores: list[dict[str, Any]],
    sales: list[dict[str, Any]],
    sale_items: list[dict[str, Any]],
    customer_returns: list[dict[str, Any]],
    dataset_cutoff: date,
) -> dict[str, int]:
    """
    Reuse the fixed shift population to cover store activity and returns.

    This is deterministic post-processing. It does not add shifts, does not
    create cross-store assignments, and does not consume global random calls.
    """
    normalized_intervals = normalize_shifts_to_store_hours(shifts, stores)
    (
        time_off_cancellations,
        overlap_cancellations,
        actual_time_corrections,
    ) = reconcile_staff_schedule(shifts, time_off_requests)

    store_by_id = {store["store_id"]: store for store in stores}
    employee_by_id = {employee["employee_id"]: employee for employee in employees}
    employees_by_store: dict[int, list[dict[str, Any]]] = defaultdict(list)
    for employee in employees:
        if employee["employment_status"] != "terminated":
            employees_by_store[employee["store_id"]].append(employee)

    approved_time_off: dict[int, list[tuple[date, date]]] = defaultdict(list)
    for request in time_off_requests:
        if request["request_status"] == "approved":
            approved_time_off[request["employee_id"]].append(
                (request["start_date"], request["end_date"])
            )

    sale_by_id = {sale["sale_id"]: sale for sale in sales}
    sale_item_by_id = {item["sale_item_id"]: item for item in sale_items}
    events: list[tuple[int, datetime, str, int]] = [
        (sale["store_id"], sale["sale_time"], "sale", sale["sale_id"])
        for sale in sales
    ]
    for return_row in customer_returns:
        sale_item = sale_item_by_id[return_row["sale_item_id"]]
        sale = sale_by_id[sale_item["sale_id"]]
        events.append(
            (
                sale["store_id"],
                return_row["return_time"],
                "return",
                return_row["return_id"],
            )
        )

    # Build a minimal, deterministic set of coverage windows from all store
    # events. Reusing arbitrary preexisting completed shifts can fragment the
    # coverage and exhaust one store's fixed shift pool even when a compact
    # schedule exists, so every coverage row is selected from the same fixed
    # population below.
    anchored_shift_ids: set[int] = set()
    uncovered_by_store_day: dict[tuple[int, date], list[datetime]] = defaultdict(list)
    for store_id, event_time, _, _ in events:
        uncovered_by_store_day[(store_id, event_time.date())].append(event_time)

    required_windows: dict[int, list[tuple[datetime, datetime]]] = defaultdict(list)
    for (store_id, _), event_times in sorted(uncovered_by_store_day.items()):
        store = store_by_id[store_id]
        opening_time = time.fromisoformat(str(store["opening_time"]))
        closing_time = time.fromisoformat(str(store["closing_time"]))
        remaining = sorted(event_times)
        while remaining:
            shift_start, shift_end = _coverage_interval(
                remaining[0], opening_time, closing_time
            )
            required_windows[store_id].append((shift_start, shift_end))
            remaining = [event_time for event_time in remaining if event_time > shift_end]

    scheduled_intervals: dict[
        int, list[tuple[datetime, datetime, int]]
    ] = defaultdict(list)
    actual_intervals: dict[
        int, list[tuple[datetime, datetime, int]]
    ] = defaultdict(list)
    for shift in shifts:
        if shift["shift_status"] != "cancelled":
            scheduled_intervals[shift["employee_id"]].append(
                (
                    shift["scheduled_start"],
                    shift["scheduled_end"],
                    shift["shift_id"],
                )
            )
        if shift["shift_status"] == "completed":
            actual_intervals[shift["employee_id"]].append(
                (shift["actual_start"], shift["actual_end"], shift["shift_id"])
            )

    def remove_shift_intervals(shift: dict[str, Any]) -> None:
        employee_id = shift["employee_id"]
        shift_id = shift["shift_id"]
        scheduled_intervals[employee_id] = [
            interval
            for interval in scheduled_intervals[employee_id]
            if interval[2] != shift_id
        ]
        actual_intervals[employee_id] = [
            interval
            for interval in actual_intervals[employee_id]
            if interval[2] != shift_id
        ]

    def employee_interval_is_free(
        employee_id: int,
        shift_start: datetime,
        shift_end: datetime,
    ) -> bool:
        employee = employee_by_id[employee_id]
        if employee["hire_date"] > shift_start.date():
            return False
        for existing_start, existing_end, _ in scheduled_intervals[employee_id]:
            if _intervals_overlap(
                shift_start, shift_end, existing_start, existing_end
            ):
                return False
        for existing_start, existing_end, _ in actual_intervals[employee_id]:
            if _intervals_overlap(
                shift_start, shift_end, existing_start, existing_end
            ):
                return False
        return True

    def employee_is_available(
        employee_id: int,
        shift_start: datetime,
        shift_end: datetime,
    ) -> bool:
        return employee_interval_is_free(employee_id, shift_start, shift_end) and not any(
            leave_start <= shift_start.date() <= leave_end
            for leave_start, leave_end in approved_time_off.get(employee_id, [])
        )

    coverage_leave_overrides = 0

    def choose_coverage_employee(
        store_id: int,
        shift_start: datetime,
        shift_end: datetime,
        preferred_employee_id: int | None,
    ) -> dict[str, Any] | None:
        nonlocal coverage_leave_overrides
        candidates = sorted(
            employees_by_store[store_id],
            key=lambda employee: (
                preferred_employee_id is not None
                and employee["employee_id"] != preferred_employee_id,
                role_priority.get(employee["role_id"], 99),
                employee["employee_id"],
            ),
        )
        available = next(
            (
                employee
                for employee in candidates
                if employee_is_available(
                    employee["employee_id"], shift_start, shift_end
                )
            ),
            None,
        )
        if available is not None:
            return available

        # A very short custom date window can place the only employee at a
        # store on generated approved leave. Resolve that synthetic request as
        # denied instead of creating a cross-store shift or violating leave.
        for employee in candidates:
            employee_id = employee["employee_id"]
            if not employee_interval_is_free(employee_id, shift_start, shift_end):
                continue
            changed = False
            for request in time_off_requests:
                if (
                    request["employee_id"] == employee_id
                    and request["request_status"] == "approved"
                    and request["start_date"]
                    <= shift_start.date()
                    <= request["end_date"]
                ):
                    request["request_status"] = "denied"
                    changed = True
                    coverage_leave_overrides += 1
            if changed:
                approved_time_off[employee_id] = [
                    (leave_start, leave_end)
                    for leave_start, leave_end in approved_time_off[employee_id]
                    if not leave_start <= shift_start.date() <= leave_end
                ]
            if employee_is_available(employee_id, shift_start, shift_end):
                return employee
        return None

    status_priority = {
        "scheduled": 0,
        "completed": 1,
        "missed": 2,
        "cancelled": 3,
    }
    role_priority = {6: 0, 2: 1, 3: 2, 1: 3, 4: 4, 5: 5}
    donor_shifts_by_store: dict[int, list[dict[str, Any]]] = defaultdict(list)
    for shift in shifts:
        if shift["shift_id"] not in anchored_shift_ids:
            donor_shifts_by_store[shift["store_id"]].append(shift)
    for store_id in donor_shifts_by_store:
        donor_shifts_by_store[store_id].sort(
            key=lambda item: (
                status_priority[item["shift_status"]],
                item["shift_id"],
            )
        )

    # Every non-anchored shift is movable. Remove those intervals before
    # assigning coverage so an obsolete donor schedule cannot block a valid
    # employee. Unused donor rows are reinserted only when they remain
    # conflict-free after the coverage schedule has been established.
    for donors in donor_shifts_by_store.values():
        for donor in donors:
            remove_shift_intervals(donor)

    coverage_shifts_retimed = 0
    coverage_shifts_reclassified = 0
    coverage_shifts_added = 0
    next_shift_id = max(shift["shift_id"] for shift in shifts) + 1
    used_donor_ids: set[int] = set()
    for store_id, windows in sorted(required_windows.items()):
        donors = donor_shifts_by_store[store_id]
        for shift_start, shift_end in windows:
            assigned = False
            for donor in donors:
                if donor["shift_id"] in used_donor_ids:
                    continue
                original_employee_id = donor["employee_id"]
                original_status = donor["shift_status"]

                available_employee = choose_coverage_employee(
                    store_id,
                    shift_start,
                    shift_end,
                    original_employee_id,
                )
                if available_employee is None:
                    continue

                donor["employee_id"] = available_employee["employee_id"]
                donor["store_id"] = store_id
                donor["scheduled_start"] = shift_start
                donor["scheduled_end"] = shift_end
                donor["actual_start"] = shift_start
                donor["actual_end"] = shift_end
                donor["shift_status"] = "completed"
                scheduled_intervals[donor["employee_id"]].append(
                    (shift_start, shift_end, donor["shift_id"])
                )
                actual_intervals[donor["employee_id"]].append(
                    (shift_start, shift_end, donor["shift_id"])
                )
                used_donor_ids.add(donor["shift_id"])
                anchored_shift_ids.add(donor["shift_id"])
                coverage_shifts_retimed += 1
                if original_status != "completed":
                    coverage_shifts_reclassified += 1
                assigned = True
                break

            if not assigned:
                available_employee = choose_coverage_employee(
                    store_id,
                    shift_start,
                    shift_end,
                    None,
                )
                if available_employee is None:
                    raise ValueError(
                        "No same-store employee is available to cover activity "
                        f"for store_id={store_id} at {shift_start}."
                    )
                added_shift = {
                    "shift_id": next_shift_id,
                    "employee_id": available_employee["employee_id"],
                    "store_id": store_id,
                    "scheduled_start": shift_start,
                    "scheduled_end": shift_end,
                    "actual_start": shift_start,
                    "actual_end": shift_end,
                    "shift_status": "completed",
                }
                shifts.append(added_shift)
                scheduled_intervals[added_shift["employee_id"]].append(
                    (shift_start, shift_end, next_shift_id)
                )
                actual_intervals[added_shift["employee_id"]].append(
                    (shift_start, shift_end, next_shift_id)
                )
                anchored_shift_ids.add(next_shift_id)
                next_shift_id += 1
                coverage_shifts_added += 1

    unused_shift_conflicts_cancelled = 0
    for donors in donor_shifts_by_store.values():
        for donor in donors:
            if donor["shift_id"] in used_donor_ids or donor["shift_status"] == "cancelled":
                continue
            employee_id = donor["employee_id"]
            scheduled_available = employee_is_available(
                employee_id,
                donor["scheduled_start"],
                donor["scheduled_end"],
            )
            actual_available = True
            if donor["shift_status"] == "completed":
                actual_available = employee_is_available(
                    employee_id,
                    donor["actual_start"],
                    donor["actual_end"],
                )
            if scheduled_available and actual_available:
                scheduled_intervals[employee_id].append(
                    (
                        donor["scheduled_start"],
                        donor["scheduled_end"],
                        donor["shift_id"],
                    )
                )
                if donor["shift_status"] == "completed":
                    actual_intervals[employee_id].append(
                        (
                            donor["actual_start"],
                            donor["actual_end"],
                            donor["shift_id"],
                        )
                    )
            else:
                donor["shift_status"] = "cancelled"
                donor["actual_start"] = None
                donor["actual_end"] = None
                unused_shift_conflicts_cancelled += 1

    cutoff_timestamp = datetime.combine(dataset_cutoff, time.max)
    historical_scheduled_resolved = 0
    for shift in shifts:
        if (
            shift["shift_status"] == "scheduled"
            and shift["scheduled_end"] <= cutoff_timestamp
        ):
            shift["shift_status"] = (
                "missed" if shift["shift_id"] % 2 else "cancelled"
            )
            shift["actual_start"] = None
            shift["actual_end"] = None
            historical_scheduled_resolved += 1

    (
        final_time_off_cancellations,
        final_overlap_cancellations,
        final_actual_time_corrections,
    ) = reconcile_staff_schedule(shifts, time_off_requests)

    completed_by_store_day = defaultdict(list)
    for shift in shifts:
        if shift["shift_status"] == "completed":
            completed_by_store_day[
                (shift["store_id"], shift["actual_start"].date())
            ].append(shift)

    uncovered_events = []
    for store_id, event_time, event_type, event_id in events:
        if not any(
            shift["actual_start"] <= event_time <= shift["actual_end"]
            for shift in completed_by_store_day.get(
                (store_id, event_time.date()), []
            )
        ):
            uncovered_events.append((event_type, event_id, store_id, event_time))
    if uncovered_events:
        raise ValueError(
            "Completed-shift coverage is missing for "
            f"{len(uncovered_events)} operational event(s)."
        )

    processor_role_priority = {6: 0, 2: 1, 1: 2, 3: 3, 4: 4, 5: 5}
    return_processors_changed = 0
    for return_row in customer_returns:
        sale_item = sale_item_by_id[return_row["sale_item_id"]]
        sale = sale_by_id[sale_item["sale_id"]]
        return_time = return_row["return_time"]
        covering_shifts = [
            shift
            for shift in completed_by_store_day[
                (sale["store_id"], return_time.date())
            ]
            if shift["actual_start"] <= return_time <= shift["actual_end"]
        ]
        processor_shift = min(
            covering_shifts,
            key=lambda shift: (
                processor_role_priority.get(
                    employee_by_id[shift["employee_id"]]["role_id"], 99
                ),
                shift["employee_id"],
                shift["shift_id"],
            ),
        )
        if return_row["processed_by_employee_id"] != processor_shift["employee_id"]:
            return_row["processed_by_employee_id"] = processor_shift["employee_id"]
            return_processors_changed += 1

    return {
        "normalized_shift_intervals": normalized_intervals,
        "initial_time_off_cancellations": time_off_cancellations,
        "initial_overlap_cancellations": overlap_cancellations,
        "initial_actual_time_corrections": actual_time_corrections,
        "coverage_shifts_retimed": coverage_shifts_retimed,
        "coverage_shifts_reclassified": coverage_shifts_reclassified,
        "coverage_shifts_added": coverage_shifts_added,
        "coverage_leave_overrides": coverage_leave_overrides,
        "unused_shift_conflicts_cancelled": unused_shift_conflicts_cancelled,
        "historical_scheduled_resolved": historical_scheduled_resolved,
        "final_time_off_cancellations": final_time_off_cancellations,
        "final_overlap_cancellations": final_overlap_cancellations,
        "final_actual_time_corrections": final_actual_time_corrections,
        "return_processors_changed": return_processors_changed,
    }


def generate_product_categories() -> list[dict[str, Any]]:
    return [
        {"category_id": idx, "category_name": category}
        for idx, category in enumerate(PRODUCT_LIBRARY.keys(), start=1)
    ]


def generate_products() -> list[dict[str, Any]]:
    rows = []
    product_id = 1
    for category_id, (_, products) in enumerate(PRODUCT_LIBRARY.items(), start=1):
        for product_name, brand, unit_size, price in products:
            rows.append(
                {
                    "product_id": product_id,
                    "category_id": category_id,
                    "product_name": product_name,
                    "brand": brand,
                    "unit_size": unit_size,
                    "retail_price": money(price),
                    "active_flag": random.random() > 0.04,
                }
            )
            product_id += 1
    return rows


def generate_store_inventory(
    stores: list[dict[str, Any]],
    products: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    rows = []
    for store in stores:
        if store["operational_status"] != "active":
            continue
        for product in products:
            if not product["active_flag"]:
                continue
            rows.append(
                {
                    "store_id": store["store_id"],
                    "product_id": product["product_id"],
                    "quantity_on_hand": 0,
                    "reorder_threshold": random.randint(8, 35),
                }
            )
    return rows


def generate_vendors() -> list[dict[str, Any]]:
    rows = []
    for idx, vendor_name in enumerate(VENDOR_NAMES, start=1):
        first = FIRST_NAMES[(idx + 3) % len(FIRST_NAMES)]
        last = LAST_NAMES[(idx + 6) % len(LAST_NAMES)]
        slug = re.sub(r"[^a-z0-9]+", "", vendor_name.lower())
        rows.append(
            {
                "vendor_id": idx,
                "vendor_name": vendor_name,
                "contact_name": f"{first} {last}",
                "email": f"orders@{slug}.com",
                "phone": f"718-555-{1000 + idx:04d}",
                "vendor_status": "active" if idx <= 9 else "inactive",
            }
        )
    return rows


def generate_vendor_products(
    vendors: list[dict[str, Any]],
    products: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    active_vendor_ids = [v["vendor_id"] for v in vendors if v["vendor_status"] == "active"]
    rows = []
    vendor_product_id = 1
    for product in products:
        vendor_count = random.choice([1, 2, 2, 3])
        for vendor_id in random.sample(active_vendor_ids, vendor_count):
            retail_price = money(product["retail_price"])
            cost = money(retail_price * Decimal(str(random.uniform(0.48, 0.75))))
            rows.append(
                {
                    "vendor_product_id": vendor_product_id,
                    "vendor_id": vendor_id,
                    "product_id": product["product_id"],
                    "unit_purchase_cost": cost,
                    "min_order_quantity": random.choice([6, 8, 10, 12, 20, 24, 36, 48]),
                    "lead_time_days": random.randint(1, 7),
                }
            )
            vendor_product_id += 1
    return rows


def calculate_opening_inventory_targets(
    store_inventory: list[dict[str, Any]],
    sales: list[dict[str, Any]],
    sale_items: list[dict[str, Any]],
    start_date: date,
) -> dict[tuple[int, int], int]:
    """Forecast opening stock from the first ten days plus reorder safety."""
    sale_by_id = {sale["sale_id"]: sale for sale in sales}
    opening_window_end = start_date + timedelta(days=OPENING_INVENTORY_DAYS)
    opening_demand: dict[tuple[int, int], int] = defaultdict(int)
    for item in sale_items:
        sale = sale_by_id[item["sale_id"]]
        if sale["sale_time"].date() < opening_window_end:
            opening_demand[(sale["store_id"], item["product_id"])] += int(
                item["quantity_sold"]
            )

    return {
        (row["store_id"], row["product_id"]): max(
            int(row["reorder_threshold"]) + REORDER_SAFETY_UNITS,
            math.ceil(
                opening_demand.get((row["store_id"], row["product_id"]), 0)
                * 1.20
            ),
        )
        for row in store_inventory
    }


def generate_purchase_data(
    stores: list[dict[str, Any]],
    vendor_products: list[dict[str, Any]],
    store_inventory: list[dict[str, Any]],
    sales: list[dict[str, Any]],
    sale_items: list[dict[str, Any]],
    start_date: date,
    end_date: date,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]], list[dict[str, Any]], list[dict[str, Any]]]:
    """Build weekly demand-driven replenishment for the complete population."""
    active_store_ids = [
        store["store_id"]
        for store in stores
        if store["operational_status"] == "active"
    ]
    vendor_product_by_vendor: dict[int, list[dict[str, Any]]] = defaultdict(list)
    vendor_product_by_product: dict[int, list[dict[str, Any]]] = defaultdict(list)
    for vp in vendor_products:
        vendor_product_by_vendor[vp["vendor_id"]].append(vp)
        vendor_product_by_product[vp["product_id"]].append(vp)
    for rows in vendor_product_by_vendor.values():
        rows.sort(key=lambda row: row["vendor_product_id"])
    for rows in vendor_product_by_product.values():
        rows.sort(key=lambda row: row["vendor_product_id"])

    sale_by_id = {sale["sale_id"]: sale for sale in sales}
    weekly_demand: dict[tuple[int, int, int], int] = defaultdict(int)
    for item in sale_items:
        sale = sale_by_id[item["sale_id"]]
        week_index = (sale["sale_time"].date() - start_date).days // REPLENISHMENT_DAYS
        weekly_demand[(sale["store_id"], week_index, item["product_id"])] += int(
            item["quantity_sold"]
        )

    opening_targets = calculate_opening_inventory_targets(
        store_inventory, sales, sale_items, start_date
    )
    reorder_floor = {
        (row["store_id"], row["product_id"]): (
            int(row["reorder_threshold"]) + REORDER_SAFETY_UNITS
        )
        for row in store_inventory
    }
    projected_balance = dict(opening_targets)
    product_ids_by_store: dict[int, list[int]] = defaultdict(list)
    for row in store_inventory:
        product_ids_by_store[row["store_id"]].append(row["product_id"])

    grouped_replenishment: dict[
        tuple[int, int, int], list[tuple[dict[str, Any], int]]
    ] = defaultdict(list)
    maximum_week_index = (end_date - start_date).days // REPLENISHMENT_DAYS
    for week_index in range(maximum_week_index + 1):
        for store_id in sorted(active_store_ids):
            for product_id in sorted(product_ids_by_store[store_id]):
                key = (store_id, product_id)
                demand_units = weekly_demand.get(
                    (store_id, week_index, product_id), 0
                )
                required_units = max(
                    0,
                    demand_units
                    + reorder_floor[key]
                    - projected_balance[key],
                )
                if required_units > 0:
                    target_units = math.ceil(
                        required_units
                        * (100 + PURCHASE_SAFETY_PERCENT)
                        / 100
                    )
                    options = vendor_product_by_product[product_id]
                    if not options:
                        raise ValueError(
                            "No vendor product exists for sold "
                            f"product_id={product_id}."
                        )
                    scored_options = []
                    for option in options:
                        minimum = int(option["min_order_quantity"])
                        ordered_units = (
                            math.ceil(max(target_units, minimum) / minimum)
                            * minimum
                        )
                        scored_options.append(
                            (
                                money(
                                    ordered_units
                                    * Decimal(option["unit_purchase_cost"])
                                ),
                                ordered_units - target_units,
                                option["vendor_product_id"],
                                ordered_units,
                                option,
                            )
                        )
                    scored_options.sort(key=lambda row: row[:3])
                    shortlist = scored_options[: min(2, len(scored_options))]
                    chosen = shortlist[
                        (store_id + product_id + week_index) % len(shortlist)
                    ]
                    ordered_units = chosen[3]
                    selected_vendor_product = chosen[4]
                    grouped_replenishment[
                        (
                            store_id,
                            week_index,
                            selected_vendor_product["vendor_id"],
                        )
                    ].append((selected_vendor_product, ordered_units))
                    projected_balance[key] += ordered_units
                projected_balance[key] -= demand_units

    purchase_orders = []
    purchase_order_items = []
    deliveries = []
    delivery_adjustments = []

    po_id = 1
    poi_id = 1
    delivery_id = 1
    adjustment_id = 1

    def append_order(
        store_id: int,
        vendor_id: int,
        order_date: date,
        selected: list[tuple[dict[str, Any], int]],
        status: str,
    ) -> None:
        nonlocal po_id, poi_id, delivery_id, adjustment_id
        selected_lead_time = max(
            vendor_product["lead_time_days"]
            for vendor_product, _ in selected
        )
        expected_date = order_date + timedelta(days=selected_lead_time)
        received_date: date | None = None
        if status in {"received", "partially_received"}:
            arrival_offsets = [-1, 0, 0, 0, 1, 2]
            arrival_offset = arrival_offsets[(po_id - 1) % len(arrival_offsets)]
            received_date = max(
                order_date,
                min(end_date, expected_date + timedelta(days=arrival_offset)),
            )

        purchase_orders.append(
            {
                "purchase_order_id": po_id,
                "vendor_id": vendor_id,
                "store_id": store_id,
                "order_date": order_date,
                "expected_date": expected_date,
                "received_date": received_date,
                "order_status": status,
            }
        )

        received_summaries: list[tuple[int, int]] = []
        for vendor_product, planned_order_units in selected:
            minimum = int(vendor_product["min_order_quantity"])
            ordered = math.ceil(
                max(planned_order_units, minimum) / minimum
            ) * minimum
            if status == "partially_received":
                missing = max(1, min(3, ordered // 20))
                received = ordered - missing
            elif status == "received":
                missing = 0
                received = ordered
            else:
                received = missing = 0
            damaged = (
                1
                if received > 1
                and status in {"received", "partially_received"}
                and poi_id % 37 == 0
                else 0
            )

            purchase_order_items.append(
                {
                    "purchase_order_item_id": poi_id,
                    "purchase_order_id": po_id,
                    "vendor_product_id": vendor_product["vendor_product_id"],
                    "quantity_ordered": ordered,
                    "quantity_received": received,
                    "quantity_damaged": damaged,
                    "quantity_missing": missing,
                    "unit_purchase_cost": vendor_product["unit_purchase_cost"],
                }
            )
            if received_date is not None and received - damaged > 0:
                received_summaries.append(
                    (vendor_product["product_id"], received - damaged)
                )
            poi_id += 1

        if received_date is not None:
            deliveries.append(
                {
                    "delivery_id": delivery_id,
                    "purchase_order_id": po_id,
                    "delivery_date": received_date,
                    "delivery_status": (
                        "partial" if status == "partially_received" else "received"
                    ),
                    "notes": "Demand-driven synthetic replenishment delivery",
                }
            )
            delivery_id += 1
            for product_id, good_units in received_summaries:
                delivery_adjustments.append(
                    {
                        "adjustment_id": adjustment_id,
                        "store_id": store_id,
                        "product_id": product_id,
                        "adjustment_time": datetime.combine(
                            received_date, time(7, 30)
                        ),
                        "adjustment_type": "delivery",
                        "quantity_delta": good_units,
                        "notes": f"Inventory received for purchase order {po_id}",
                    }
                )
                adjustment_id += 1
        po_id += 1

    # One replenishment PO carries at most six items, which keeps orders
    # operationally readable while tying every received quantity to the demand
    # week it supports.
    for (store_id, week_index, vendor_id), selected_rows in sorted(
        grouped_replenishment.items()
    ):
        period_start = start_date + timedelta(
            days=week_index * REPLENISHMENT_DAYS
        )
        selected_rows.sort(key=lambda pair: pair[0]["vendor_product_id"])
        for offset in range(0, len(selected_rows), 6):
            chunk = selected_rows[offset : offset + 6]
            selected_max_lead = max(
                vendor_product["lead_time_days"]
                for vendor_product, _ in chunk
            )
            order_date = max(
                start_date,
                period_start - timedelta(days=selected_max_lead),
            )
            status = "partially_received" if po_id % 17 == 0 else "received"
            append_order(store_id, vendor_id, order_date, chunk, status)

    # A small, explicit cancelled population represents routine supplier
    # substitutions. These rows never contribute received units or inventory.
    month_cursor = date(start_date.year, start_date.month, 1)
    while month_cursor <= end_date:
        month_end = (
            date(month_cursor.year + 1, 1, 1)
            if month_cursor.month == 12
            else date(month_cursor.year, month_cursor.month + 1, 1)
        ) - timedelta(days=1)
        cancellation_date = min(end_date, max(start_date, month_cursor) + timedelta(days=11))
        cancellation_date = min(cancellation_date, month_end)
        vendor_ids = sorted(vendor_product_by_vendor)
        for store_id in active_store_ids:
            vendor_id = vendor_ids[(store_id + month_cursor.month) % len(vendor_ids)]
            options = vendor_product_by_vendor[vendor_id]
            selected = [
                (option, int(option["min_order_quantity"]))
                for option in options[: min(2, len(options))]
            ]
            append_order(
                store_id,
                vendor_id,
                cancellation_date,
                selected,
                "cancelled",
            )
        month_cursor = (
            date(month_cursor.year + 1, 1, 1)
            if month_cursor.month == 12
            else date(month_cursor.year, month_cursor.month + 1, 1)
        )

    # Three near-cutoff replenishment orders are genuinely open. Their maximum
    # selected-item lead time extends beyond the cutoff and is never shortened.
    open_vendor_id = max(
        vendor_product_by_vendor,
        key=lambda vendor_id: max(
            row["lead_time_days"]
            for row in vendor_product_by_vendor[vendor_id]
        ),
    )
    open_options = sorted(
        vendor_product_by_vendor[open_vendor_id],
        key=lambda row: (-row["lead_time_days"], row["vendor_product_id"]),
    )
    for store_id in active_store_ids:
        order_date = end_date - timedelta(days=3 - store_id)
        selected = [
            (option, int(option["min_order_quantity"]))
            for option in open_options[: min(3, len(open_options))]
        ]
        append_order(store_id, open_vendor_id, order_date, selected, "submitted")

    return purchase_orders, purchase_order_items, deliveries, delivery_adjustments


def generate_initial_inventory_adjustments(
    store_inventory: list[dict[str, Any]],
    sales: list[dict[str, Any]],
    sale_items: list[dict[str, Any]],
    start_date: date,
    starting_adjustment_id: int,
) -> list[dict[str, Any]]:
    """Create a defensible opening stock based on the first ten sales days."""
    rows = []
    adjustment_id = starting_adjustment_id
    initial_time = datetime.combine(start_date, time(7, 0))
    opening_targets = calculate_opening_inventory_targets(
        store_inventory, sales, sale_items, start_date
    )
    for inventory_row in store_inventory:
        key = (inventory_row["store_id"], inventory_row["product_id"])
        opening_units = opening_targets[key]
        rows.append(
            {
                "adjustment_id": adjustment_id,
                "store_id": inventory_row["store_id"],
                "product_id": inventory_row["product_id"],
                "adjustment_time": initial_time,
                "adjustment_type": "manual_count",
                "quantity_delta": opening_units,
                "notes": (
                    "Opening inventory based on first 10 days of complete-population demand"
                ),
            }
        )
        adjustment_id += 1
    return rows


def generate_sales_data(
    stores: list[dict[str, Any]],
    products: list[dict[str, Any]],
    employees: list[dict[str, Any]],
    time_off_requests: list[dict[str, Any]],
    sales_count: int,
    start_date: date,
    end_date: date,
) -> tuple[
    list[dict[str, Any]],
    list[dict[str, Any]],
    list[dict[str, Any]],
    list[dict[str, Any]],
    list[dict[str, Any]],
]:
    active_stores = {
        s["store_id"]: s for s in stores if s["operational_status"] == "active"
    }
    active_store_ids = list(active_stores)
    active_products = [p for p in products if p["active_flag"]]
    employees_by_store: dict[int, list[dict[str, Any]]] = defaultdict(list)
    for employee in employees:
        if employee["employment_status"] == "active":
            employees_by_store[employee["store_id"]].append(employee)
    approved_time_off_by_employee: dict[int, list[tuple[date, date]]] = defaultdict(list)
    for request in time_off_requests:
        if request["request_status"] == "approved":
            approved_time_off_by_employee[request["employee_id"]].append(
                (request["start_date"], request["end_date"])
            )

    sales_transactions = []
    sales_items = []
    payments = []
    customer_returns = []
    return_adjustments = []

    sale_item_lookup: dict[int, dict[str, Any]] = {}
    sale_item_id = 1
    payment_id = 1

    for sale_id in range(1, sales_count + 1):
        store_id = random.choice(active_store_ids)
        store = active_stores[store_id]
        sale_time = random_datetime(start_date, end_date, 8, 21)
        opening_time = time.fromisoformat(str(store["opening_time"]))
        if sale_time.time() < opening_time:
            sale_time = datetime.combine(sale_time.date(), opening_time).replace(
                minute=sale_time.minute
            )
        status = random.choices(
            ["completed", "partially_refunded", "refunded", "voided"],
            weights=[92, 4, 2, 2],
            k=1,
        )[0]

        selected = random.sample(active_products, k=random.randint(1, min(6, len(active_products))))
        subtotal = Decimal("0")
        discount_total = Decimal("0")
        line_rows = []

        for product in selected:
            quantity = random.randint(1, 4)
            unit_price = money(product["retail_price"])
            gross_line = money(unit_price * quantity)
            discount = Decimal("0")
            if random.random() < 0.18:
                discount = money(min(gross_line, Decimal(str(random.choice([0.50, 1.00, 1.50, 2.00])))))
            subtotal += gross_line
            discount_total += discount
            line_rows.append(
                {
                    "sale_item_id": sale_item_id,
                    "sale_id": sale_id,
                    "product_id": product["product_id"],
                    "quantity_sold": quantity,
                    "unit_retail_price": unit_price,
                    "item_discount": discount,
                }
            )
            sale_item_lookup[sale_item_id] = {
                **line_rows[-1],
                "store_id": store_id,
                "sale_time": sale_time,
            }
            sale_item_id += 1

        subtotal = money(subtotal)
        discount_total = money(discount_total)
        taxable = max(Decimal("0"), subtotal - discount_total)
        tax = money(taxable * TAX_RATE)
        total = money(taxable + tax)

        if status == "voided":
            # Keep line items for auditing, but no payment. The schema trigger would
            # reduce inventory, so voided transactions are omitted from sales_items.
            line_rows = []
            subtotal = discount_total = tax = total = Decimal("0")
            refund = Decimal("0")
        elif status == "refunded":
            refund = total
        elif status == "partially_refunded":
            refund = money(total * Decimal(str(random.uniform(0.15, 0.50))))
        else:
            refund = Decimal("0")

        sales_transactions.append(
            {
                "sale_id": sale_id,
                "store_id": store_id,
                "sale_time": sale_time,
                "sale_status": status,
                "subtotal_amount": subtotal,
                "discount_amount": discount_total,
                "tax_amount": tax,
                "refund_amount": refund,
                "total_amount": total,
            }
        )
        sales_items.extend(line_rows)

        if status != "voided" and total > 0:
            if random.random() < 0.06 and total >= Decimal("4.00"):
                first_amount = money(total * Decimal("0.40"))
                second_amount = money(total - first_amount)
                methods = random.sample(
                    ["cash", "credit_card", "debit_card", "mobile_pay", "gift_card"], 2
                )
                for amount, method in [(first_amount, methods[0]), (second_amount, methods[1])]:
                    payments.append(
                        {
                            "payment_id": payment_id,
                            "sale_id": sale_id,
                            "payment_method": method,
                            "payment_amount": amount,
                            "payment_time": sale_time + timedelta(minutes=1),
                        }
                    )
                    payment_id += 1
            else:
                payments.append(
                    {
                        "payment_id": payment_id,
                        "sale_id": sale_id,
                        "payment_method": random.choice(
                            ["cash", "credit_card", "debit_card", "mobile_pay", "gift_card"]
                        ),
                        "payment_amount": total,
                        "payment_time": sale_time + timedelta(minutes=1),
                    }
                )
                payment_id += 1

    sale_by_id = {row["sale_id"]: row for row in sales_transactions}

    # Create returns from non-voided sales items.
    eligible_items = [
        item for item in sales_items
        if item["quantity_sold"] > 0
        and employees_by_store.get(
            sale_item_lookup[item["sale_item_id"]]["store_id"]
        )
    ]
    return_count = min(max(25, sales_count // 50), len(eligible_items))
    for item in random.sample(eligible_items, return_count):
        lookup = sale_item_lookup[item["sale_item_id"]]
        store_id = lookup["store_id"]
        store = active_stores[store_id]
        opening_time = time.fromisoformat(str(store["opening_time"]))
        closing_time = time.fromisoformat(str(store["closing_time"]))
        raw_return_time = lookup["sale_time"] + timedelta(
            days=random.randint(1, 21),
            hours=random.randint(0, 6),
        )
        return_date = min(raw_return_time.date(), end_date)
        return_time = datetime.combine(return_date, raw_return_time.time())
        opening_datetime = datetime.combine(return_date, opening_time)
        closing_datetime = datetime.combine(return_date, closing_time)
        if return_time < opening_datetime:
            return_time = opening_datetime
        elif return_time >= closing_datetime:
            if return_date < end_date:
                return_date += timedelta(days=1)
                return_time = datetime.combine(return_date, opening_time)
            else:
                return_time = closing_datetime - timedelta(minutes=5)
        if return_time <= lookup["sale_time"]:
            return_time = min(
                closing_datetime - timedelta(seconds=1),
                lookup["sale_time"] + timedelta(minutes=1),
            )

        quantity_returned = random.randint(1, item["quantity_sold"])
        sale = sale_by_id[item["sale_id"]]
        item_net_amount = money(
            money(item["unit_retail_price"]) * item["quantity_sold"]
            - money(item["item_discount"])
        )
        sale_net_before_tax = money(
            money(sale["subtotal_amount"]) - money(sale["discount_amount"])
        )
        returned_item_net = (
            item_net_amount
            * Decimal(str(quantity_returned))
            / Decimal(str(item["quantity_sold"]))
        )
        refund_amount = money(
            returned_item_net
            * money(sale["total_amount"])
            / sale_net_before_tax
        )
        eligible_processors = [
            employee
            for employee in employees_by_store[store_id]
            if employee["hire_date"] <= return_time.date()
            and not any(
                start_date <= return_time.date() <= end_date
                for start_date, end_date in approved_time_off_by_employee.get(
                    employee["employee_id"], []
                )
            )
        ]
        if not eligible_processors:
            continue
        employee = random.choice(employees_by_store[store_id])
        if employee not in eligible_processors:
            employee = min(
                eligible_processors,
                key=lambda item: (item["hire_date"], item["employee_id"]),
            )
        return_id = len(customer_returns) + 1
        reason = random.choice(
            ["Changed mind", "Damaged packaging", "Incorrect item", "Quality issue", "Product not needed"]
        )
        customer_returns.append(
            {
                "return_id": return_id,
                "sale_item_id": item["sale_item_id"],
                "processed_by_employee_id": employee["employee_id"],
                "return_time": return_time,
                "quantity_returned": quantity_returned,
                "refund_amount": refund_amount,
                "return_reason": reason,
            }
        )
        # Only some returned products are resellable.
        if reason in {"Changed mind", "Incorrect item", "Product not needed"}:
            return_adjustments.append(
                {
                    "store_id": store_id,
                    "product_id": item["product_id"],
                    "adjustment_time": return_time,
                    "adjustment_type": "return",
                    "quantity_delta": quantity_returned,
                    "notes": f"Resellable customer return {return_id}",
                }
            )

    # Keep transaction refund fields synchronized with the detailed return
    # records. Payments still represent the amount originally collected.
    returns_by_sale: dict[int, list[dict[str, Any]]] = defaultdict(list)
    for return_row in customer_returns:
        sale_id = sale_item_lookup[return_row["sale_item_id"]]["sale_id"]
        returns_by_sale[sale_id].append(return_row)

    for sale in sales_transactions:
        if sale["sale_status"] == "voided":
            continue
        sale_returns = returns_by_sale.get(sale["sale_id"], [])
        if not sale_returns:
            sale["sale_status"] = "completed"
            sale["refund_amount"] = Decimal("0")
            continue

        total_return_refund = money(
            sum((money(row["refund_amount"]) for row in sale_returns), Decimal("0"))
        )
        if total_return_refund > money(sale["total_amount"]):
            excess = total_return_refund - money(sale["total_amount"])
            sale_returns[-1]["refund_amount"] = money(
                money(sale_returns[-1]["refund_amount"]) - excess
            )
            total_return_refund = money(sale["total_amount"])

        sale["refund_amount"] = total_return_refund
        if total_return_refund == Decimal("0"):
            # A fully discounted item can be physically returned without a
            # monetary refund. The transaction remains completed.
            sale["sale_status"] = "completed"
        elif total_return_refund == money(sale["total_amount"]):
            sale["sale_status"] = "refunded"
        else:
            sale["sale_status"] = "partially_refunded"

    return sales_transactions, sales_items, payments, customer_returns, return_adjustments


def generate_other_inventory_adjustments(
    store_inventory: list[dict[str, Any]],
    start_date: date,
    end_date: date,
) -> list[dict[str, Any]]:
    rows = []
    candidate_count = max(20, len(store_inventory) // 3)
    for inventory in random.sample(store_inventory, min(candidate_count, len(store_inventory))):
        adjustment_type = random.choice(["damage", "expired"])
        rows.append(
            {
                "store_id": inventory["store_id"],
                "product_id": inventory["product_id"],
                "adjustment_time": random_datetime(start_date, end_date),
                "adjustment_type": adjustment_type,
                "quantity_delta": -random.randint(1, 3),
                "notes": f"Synthetic {adjustment_type} adjustment",
            }
        )
    return rows


def assert_sufficient_inventory_population(
    store_inventory: list[dict[str, Any]],
    initial_adjustments: list[dict[str, Any]],
    delivery_adjustments: list[dict[str, Any]],
    sales: list[dict[str, Any]],
    sale_items: list[dict[str, Any]],
    post_sale_adjustments: list[dict[str, Any]],
) -> tuple[int, int]:
    """
    Reject a population that relies on an after-the-fact opening-stock top-up.

    Opening inventory is forecast-based and purchasing is demand-driven before
    this function runs. Both the real event timeline and the database loader's
    trigger order must therefore be nonnegative without mutating any row here.
    """
    starting_balance: dict[tuple[int, int], int] = defaultdict(int)
    initial_row_by_key: dict[tuple[int, int], dict[str, Any]] = {}

    for row in store_inventory:
        key = (row["store_id"], row["product_id"])
        starting_balance[key] += int(row["quantity_on_hand"])

    for row in initial_adjustments:
        key = (row["store_id"], row["product_id"])
        starting_balance[key] += int(row["quantity_delta"])
        initial_row_by_key[key] = row

    sale_store = {row["sale_id"]: row["store_id"] for row in sales}
    sale_time = {row["sale_id"]: row["sale_time"] for row in sales}

    chronological_events: list[
        tuple[datetime, int, int, tuple[int, int], int]
    ] = []
    loader_events: list[tuple[tuple[int, int], int]] = []

    for row in sorted(
        delivery_adjustments,
        key=lambda item: (item["adjustment_time"], item["adjustment_id"]),
    ):
        key = (row["store_id"], row["product_id"])
        chronological_events.append(
            (
                row["adjustment_time"],
                0,
                row["adjustment_id"],
                key,
                int(row["quantity_delta"]),
            )
        )
        loader_events.append((key, int(row["quantity_delta"])))

    for row in sale_items:
        key = (sale_store[row["sale_id"]], row["product_id"])
        quantity_delta = -int(row["quantity_sold"])
        chronological_events.append(
            (
                sale_time[row["sale_id"]],
                1,
                row["sale_item_id"],
                key,
                quantity_delta,
            )
        )
        loader_events.append((key, quantity_delta))

    for row in sorted(
        post_sale_adjustments,
        key=lambda item: (item["adjustment_time"], item["adjustment_id"]),
    ):
        key = (row["store_id"], row["product_id"])
        chronological_events.append(
            (
                row["adjustment_time"],
                2,
                row["adjustment_id"],
                key,
                int(row["quantity_delta"]),
            )
        )
        loader_events.append((key, int(row["quantity_delta"])))

    def top_up_required(
        events: Iterable[tuple[tuple[int, int], int]],
    ) -> dict[tuple[int, int], int]:
        running = defaultdict(int, starting_balance)
        minimum = defaultdict(int, starting_balance)
        for key, quantity_delta in events:
            running[key] += quantity_delta
            minimum[key] = min(minimum[key], running[key])
        return {key: -balance for key, balance in minimum.items() if balance < 0}

    chronological_required = top_up_required(
        (key, quantity_delta)
        for _, _, _, key, quantity_delta in sorted(chronological_events)
    )
    loader_required = top_up_required(loader_events)

    required = {
        key: max(
            chronological_required.get(key, 0),
            loader_required.get(key, 0),
        )
        for key in set(chronological_required) | set(loader_required)
        if max(
            chronological_required.get(key, 0),
            loader_required.get(key, 0),
        ) > 0
    }
    if required:
        preview = ", ".join(
            f"store={store_id}/product={product_id}: {units} unit(s)"
            for (store_id, product_id), units in sorted(required.items())[:10]
        )
        raise ValueError(
            "Demand-driven purchasing failed inventory sufficiency; no manual "
            f"top-up was applied. Required: {preview}"
        )

    return 0, 0


def generate_operating_expenses(
    stores: list[dict[str, Any]],
    shifts: list[dict[str, Any]],
    employees: list[dict[str, Any]],
    start_date: date,
    end_date: date,
) -> list[dict[str, Any]]:
    rows = []
    expense_id = 1
    active_stores = [s for s in stores if s["operational_status"] == "active"]
    employee_by_id = {e["employee_id"]: e for e in employees}

    month_cursor = date(start_date.year, start_date.month, 1)
    end_month = date(end_date.year, end_date.month, 1)

    while month_cursor <= end_month:
        next_month = (
            date(month_cursor.year + 1, 1, 1)
            if month_cursor.month == 12
            else date(month_cursor.year, month_cursor.month + 1, 1)
        )
        month_end = min(end_date, next_month - timedelta(days=1))
        for store in active_stores:
            store_id = store["store_id"]
            fixed_expenses = [
                ("rent", money(random.uniform(6500, 9500)), "Monthly store rent"),
                ("utilities", money(random.uniform(700, 1400)), "Electricity, water, and internet"),
                ("supplies", money(random.uniform(150, 450)), "Cleaning and office supplies"),
            ]
            for expense_type, amount, description in fixed_expenses:
                rows.append(
                    {
                        "expense_id": expense_id,
                        "store_id": store_id,
                        "expense_date": min(month_end, month_cursor + timedelta(days=random.randint(0, 10))),
                        "expense_type": expense_type,
                        "amount": amount,
                        "description": description,
                    }
                )
                expense_id += 1

            monthly_shifts = [
                shift for shift in shifts
                if shift["store_id"] == store_id
                and shift["shift_status"] == "completed"
                and month_cursor <= shift["scheduled_start"].date() <= month_end
                and shift["actual_start"] is not None
                and shift["actual_end"] is not None
            ]
            wage_total = Decimal("0")
            for shift in monthly_shifts:
                hours = Decimal(
                    str((shift["actual_end"] - shift["actual_start"]).total_seconds() / 3600)
                )
                wage_total += hours * money(employee_by_id[shift["employee_id"]]["hourly_pay_rate"])
            if wage_total > 0:
                rows.append(
                    {
                        "expense_id": expense_id,
                        "store_id": store_id,
                        "expense_date": month_end,
                        "expense_type": "wages",
                        "amount": money(wage_total),
                        "description": "Calculated from completed synthetic staff shifts",
                    }
                )
                expense_id += 1

            if random.random() < 0.35:
                expense_type = random.choice(["maintenance", "marketing", "other"])
                rows.append(
                    {
                        "expense_id": expense_id,
                        "store_id": store_id,
                        "expense_date": random_date(month_cursor, month_end),
                        "expense_type": expense_type,
                        "amount": money(random.uniform(150, 1800)),
                        "description": f"Synthetic {expense_type} expense",
                    }
                )
                expense_id += 1
        month_cursor = next_month
    return rows


def generate_all_data(config: PipelineConfig) -> None:
    # Recreate the locked master/reference population exactly as it appeared in
    # Option A. The discarded reference shifts and leave requests exist only to
    # advance that historical protected-data random stream.
    random.seed(RANDOM_SEED)
    output_dir = config.output_dir
    output_dir.mkdir(parents=True, exist_ok=True)

    stores = generate_stores()
    roles = generate_job_roles()
    employees = generate_employees(config.employee_count, config.end_date)

    generate_staff_shifts(
        employees,
        PROTECTED_REFERENCE_START_DATE,
        config.end_date,
    )
    generate_time_off_requests(
        employees,
        PROTECTED_REFERENCE_START_DATE,
        config.end_date,
    )
    categories = generate_product_categories()
    products = generate_products()
    inventory = generate_store_inventory(stores, products)
    vendors = generate_vendors()
    vendor_products = generate_vendor_products(vendors, products)

    seed_fact_domain("staffing")
    shifts = generate_staff_shifts(employees, config.start_date, config.end_date)
    time_off = generate_time_off_requests(employees, config.start_date, config.end_date)
    denied_requests, cancelled_requests = reconcile_time_off_lifecycle(
        time_off,
        config.end_date,
    )
    logging.info(
        "Resolved historical pending time off: %d denied, %d cancelled",
        denied_requests,
        cancelled_requests,
    )
    (
        time_off_cancellations,
        overlap_cancellations,
        actual_time_corrections,
    ) = reconcile_staff_schedule(
        shifts,
        time_off,
    )
    logging.info(
        "Reconciled staff schedule: %d approved-time-off cancellation(s), "
        "%d overlap cancellation(s), %d actual-time correction(s)",
        time_off_cancellations,
        overlap_cancellations,
        actual_time_corrections,
    )

    seed_fact_domain("sales")
    sales, sale_items, payments, returns, return_adjustments = generate_sales_data(
        stores,
        products,
        employees,
        time_off,
        config.sales_count,
        config.start_date,
        config.end_date,
    )

    staffing_results = reconcile_operational_staffing(
        shifts,
        time_off,
        employees,
        stores,
        sales,
        sale_items,
        returns,
        config.end_date,
    )
    logging.info(
        "Reconciled operational staffing: %d coverage shift(s) retimed, "
        "%d added for custom-volume coverage, %d leave request(s) resolved, "
        "%d stale scheduled shift(s) resolved, %d return processor(s) changed",
        staffing_results["coverage_shifts_retimed"],
        staffing_results["coverage_shifts_added"],
        staffing_results["coverage_leave_overrides"],
        staffing_results["historical_scheduled_resolved"],
        staffing_results["return_processors_changed"],
    )

    seed_fact_domain("purchasing")
    po, poi, deliveries, delivery_adjustments = generate_purchase_data(
        stores,
        vendor_products,
        inventory,
        sales,
        sale_items,
        config.start_date,
        config.end_date,
    )

    initial_adjustments = generate_initial_inventory_adjustments(
        inventory,
        sales,
        sale_items,
        config.start_date,
        starting_adjustment_id=1,
    )
    next_adjustment_id = len(initial_adjustments) + 1

    for row in delivery_adjustments:
        row["adjustment_id"] = next_adjustment_id
        next_adjustment_id += 1

    for row in return_adjustments:
        row["adjustment_id"] = next_adjustment_id
        next_adjustment_id += 1

    seed_fact_domain("inventory_loss")
    other_adjustments = generate_other_inventory_adjustments(
        inventory, config.start_date, config.end_date
    )
    for row in other_adjustments:
        row["adjustment_id"] = next_adjustment_id
        next_adjustment_id += 1

    adjusted_inventory_rows, added_inventory_units = assert_sufficient_inventory_population(
        inventory,
        initial_adjustments,
        delivery_adjustments,
        sales,
        sale_items,
        return_adjustments + other_adjustments,
    )
    logging.info(
        "Validated opening inventory without emergency top-ups: %d row(s), "
        "%d unit(s) added",
        adjusted_inventory_rows,
        added_inventory_units,
    )

    adjustments = (
        initial_adjustments
        + delivery_adjustments
        + return_adjustments
        + other_adjustments
    )
    seed_fact_domain("expenses")
    expenses = generate_operating_expenses(
        stores, shifts, employees, config.start_date, config.end_date
    )

    datasets = {
        "stores": stores,
        "job_roles": roles,
        "employees": employees,
        "staff_shifts": shifts,
        "time_off_requests": time_off,
        "product_categories": categories,
        "products": products,
        "store_inventory": inventory,
        "inventory_adjustments": adjustments,
        "vendors": vendors,
        "vendor_products": vendor_products,
        "purchase_orders": po,
        "purchase_order_items": poi,
        "deliveries": deliveries,
        "sales_transactions": sales,
        "sales_items": sale_items,
        "customer_returns": returns,
        "payments": payments,
        "operating_expenses": expenses,
    }

    for table_name in TABLE_ORDER:
        write_csv(output_dir / f"{table_name}.csv", datasets[table_name])
        logging.info("Generated %-24s %8d rows", table_name, len(datasets[table_name]))


# ---------------------------------------------------------------------------
# Transformation and validation
# ---------------------------------------------------------------------------

DATE_COLUMNS = {
    "employees": ["hire_date"],
    "time_off_requests": ["request_date", "start_date", "end_date", "review_date"],
    "purchase_orders": ["order_date", "expected_date", "received_date"],
    "deliveries": ["delivery_date"],
    "operating_expenses": ["expense_date"],
}

TIMESTAMP_COLUMNS = {
    "staff_shifts": ["scheduled_start", "scheduled_end", "actual_start", "actual_end"],
    "inventory_adjustments": ["adjustment_time"],
    "sales_transactions": ["sale_time"],
    "customer_returns": ["return_time"],
    "payments": ["payment_time"],
}

NUMERIC_COLUMNS = {
    "employees": ["hourly_pay_rate"],
    "products": ["retail_price"],
    "vendor_products": ["unit_purchase_cost"],
    "purchase_order_items": ["unit_purchase_cost"],
    "sales_transactions": [
        "subtotal_amount", "discount_amount", "tax_amount",
        "refund_amount", "total_amount",
    ],
    "sales_items": ["unit_retail_price", "item_discount"],
    "customer_returns": ["refund_amount"],
    "payments": ["payment_amount"],
    "operating_expenses": ["amount"],
}

INTEGER_COLUMNS = {
    "stores": ["store_id"],
    "job_roles": ["role_id"],
    "employees": ["employee_id", "store_id", "role_id"],
    "staff_shifts": ["shift_id", "employee_id", "store_id"],
    "time_off_requests": ["request_id", "employee_id"],
    "product_categories": ["category_id"],
    "products": ["product_id", "category_id"],
    "store_inventory": ["store_id", "product_id", "quantity_on_hand", "reorder_threshold"],
    "inventory_adjustments": [
        "adjustment_id", "store_id", "product_id", "quantity_delta"
    ],
    "vendors": ["vendor_id"],
    "vendor_products": [
        "vendor_product_id", "vendor_id", "product_id",
        "min_order_quantity", "lead_time_days",
    ],
    "purchase_orders": ["purchase_order_id", "vendor_id", "store_id"],
    "purchase_order_items": [
        "purchase_order_item_id", "purchase_order_id", "vendor_product_id",
        "quantity_ordered", "quantity_received", "quantity_damaged", "quantity_missing",
    ],
    "deliveries": ["delivery_id", "purchase_order_id"],
    "sales_transactions": ["sale_id", "store_id"],
    "sales_items": ["sale_item_id", "sale_id", "product_id", "quantity_sold"],
    "customer_returns": [
        "return_id", "sale_item_id", "processed_by_employee_id", "quantity_returned"
    ],
    "payments": ["payment_id", "sale_id"],
    "operating_expenses": ["expense_id", "store_id"],
}

BOOLEAN_COLUMNS = {"products": ["active_flag"]}

PRIMARY_KEYS = {
    "stores": ["store_id"],
    "job_roles": ["role_id"],
    "employees": ["employee_id"],
    "staff_shifts": ["shift_id"],
    "time_off_requests": ["request_id"],
    "product_categories": ["category_id"],
    "products": ["product_id"],
    "store_inventory": ["store_id", "product_id"],
    "inventory_adjustments": ["adjustment_id"],
    "vendors": ["vendor_id"],
    "vendor_products": ["vendor_product_id"],
    "purchase_orders": ["purchase_order_id"],
    "purchase_order_items": ["purchase_order_item_id"],
    "deliveries": ["delivery_id"],
    "sales_transactions": ["sale_id"],
    "sales_items": ["sale_item_id"],
    "customer_returns": ["return_id"],
    "payments": ["payment_id"],
    "operating_expenses": ["expense_id"],
}


def transform_table(table_name: str, path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing staging file: {path}")

    raw_df = pd.read_csv(path, dtype=str)
    raw_df.columns = [
        standardize_column_name(column) for column in raw_df.columns
    ]
    df = raw_df.copy()

    # Trim text and convert common missing markers to NULL.
    for column in df.columns:
        df[column] = df[column].map(normalize_text)

    for column in INTEGER_COLUMNS.get(table_name, []):
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="raise").astype("int64")

    for column in NUMERIC_COLUMNS.get(table_name, []):
        if column in df.columns:
            df[column] = df[column].map(lambda value: None if value is None else money(value))

    for column in DATE_COLUMNS.get(table_name, []):
        if column in df.columns:
            parsed = pd.to_datetime(df[column], errors="coerce")
            raw_nonblank = raw_df[column].notna() & raw_df[column].map(
                lambda value: bool(str(value).strip()) if value is not None else False
            )
            invalid = raw_nonblank & parsed.isna()
            if invalid.any():
                invalid_rows = [
                    f"row {index + 2}: {raw_df.at[index, column]!r}"
                    for index in raw_df.index[invalid][:10]
                ]
                raise ValueError(
                    f"{table_name}.{column}: malformed nonblank date value(s): "
                    + "; ".join(invalid_rows)
                )
            df[column] = parsed.dt.date

    for column in TIMESTAMP_COLUMNS.get(table_name, []):
        if column in df.columns:
            parsed = pd.to_datetime(df[column], errors="coerce")
            raw_nonblank = raw_df[column].notna() & raw_df[column].map(
                lambda value: bool(str(value).strip()) if value is not None else False
            )
            invalid = raw_nonblank & parsed.isna()
            if invalid.any():
                invalid_rows = [
                    f"row {index + 2}: {raw_df.at[index, column]!r}"
                    for index in raw_df.index[invalid][:10]
                ]
                raise ValueError(
                    f"{table_name}.{column}: malformed nonblank timestamp "
                    "value(s): " + "; ".join(invalid_rows)
                )
            df[column] = parsed

    for column in BOOLEAN_COLUMNS.get(table_name, []):
        if column in df.columns:
            mapping = {
                "true": True, "1": True, "yes": True, "y": True,
                "false": False, "0": False, "no": False, "n": False,
            }
            df[column] = df[column].map(
                lambda value: None if value is None else mapping[str(value).lower()]
            )

    # Reject duplicate input at the source boundary. An exact duplicate is still
    # a duplicated primary key and must never disappear through silent cleanup.
    pk = PRIMARY_KEYS[table_name]
    if df[pk].isnull().any().any():
        raise ValueError(f"{table_name}: primary-key fields contain NULL values.")
    if df.duplicated(subset=pk).any():
        duplicates = df.loc[df.duplicated(subset=pk, keep=False), pk]
        raise ValueError(f"{table_name}: duplicate primary keys found:\n{duplicates.head()}")

    return df.reset_index(drop=True)


def validate_business_rules(
    data: dict[str, pd.DataFrame],
    dataset_cutoff: date,
) -> None:
    errors: list[str] = []

    # Foreign-key coverage. These checks catch broken generated snapshots before
    # the target tables are truncated.
    foreign_keys = [
        ("employees", "store_id", "stores", "store_id"),
        ("employees", "role_id", "job_roles", "role_id"),
        ("staff_shifts", "employee_id", "employees", "employee_id"),
        ("staff_shifts", "store_id", "stores", "store_id"),
        ("time_off_requests", "employee_id", "employees", "employee_id"),
        ("products", "category_id", "product_categories", "category_id"),
        ("store_inventory", "store_id", "stores", "store_id"),
        ("store_inventory", "product_id", "products", "product_id"),
        ("inventory_adjustments", "store_id", "stores", "store_id"),
        ("inventory_adjustments", "product_id", "products", "product_id"),
        ("vendor_products", "vendor_id", "vendors", "vendor_id"),
        ("vendor_products", "product_id", "products", "product_id"),
        ("purchase_orders", "vendor_id", "vendors", "vendor_id"),
        ("purchase_orders", "store_id", "stores", "store_id"),
        (
            "purchase_order_items",
            "purchase_order_id",
            "purchase_orders",
            "purchase_order_id",
        ),
        (
            "purchase_order_items",
            "vendor_product_id",
            "vendor_products",
            "vendor_product_id",
        ),
        ("deliveries", "purchase_order_id", "purchase_orders", "purchase_order_id"),
        ("sales_transactions", "store_id", "stores", "store_id"),
        ("sales_items", "sale_id", "sales_transactions", "sale_id"),
        ("sales_items", "product_id", "products", "product_id"),
        ("customer_returns", "sale_item_id", "sales_items", "sale_item_id"),
        (
            "customer_returns",
            "processed_by_employee_id",
            "employees",
            "employee_id",
        ),
        ("payments", "sale_id", "sales_transactions", "sale_id"),
        ("operating_expenses", "store_id", "stores", "store_id"),
    ]
    for child_table, child_column, parent_table, parent_column in foreign_keys:
        parent_values = set(data[parent_table][parent_column].dropna())
        missing_values = sorted(
            set(data[child_table][child_column].dropna()) - parent_values
        )
        if missing_values:
            errors.append(
                f"{child_table}.{child_column}: missing referenced values in "
                f"{parent_table}.{parent_column}: {missing_values[:10]}"
            )

    employee_hire_date = data["employees"].set_index("employee_id")[
        "hire_date"
    ].to_dict()
    for row in data["time_off_requests"].itertuples(index=False):
        if row.request_date < employee_hire_date[row.employee_id]:
            errors.append(
                f"time_off_requests: request {row.request_id} predates employee "
                f"{row.employee_id}'s hire date"
            )
        if row.start_date < row.request_date or row.end_date < row.start_date:
            errors.append(
                f"time_off_requests: invalid chronology for request {row.request_id}"
            )
        review_missing = pd.isna(row.review_date)
        if row.request_status in {"approved", "denied"} and review_missing:
            errors.append(
                f"time_off_requests: resolved request {row.request_id} lacks review date"
            )
        if row.request_status in {"pending", "cancelled"} and not review_missing:
            errors.append(
                f"time_off_requests: unresolved/cancelled request {row.request_id} "
                "has a review date"
            )
        if row.request_status == "pending" and row.end_date < dataset_cutoff:
            errors.append(
                f"time_off_requests: historical request {row.request_id} remains pending"
            )

    store_hours = {
        row.store_id: (
            time.fromisoformat(str(row.opening_time)),
            time.fromisoformat(str(row.closing_time)),
        )
        for row in data["stores"].itertuples(index=False)
    }
    for row in data["sales_transactions"].itertuples(index=False):
        opening_time, closing_time = store_hours[row.store_id]
        if not opening_time <= row.sale_time.time() < closing_time:
            errors.append(
                f"sales_transactions: sale {row.sale_id} occurs outside store hours"
            )

    # staff_shifts.store_id is the actual work location. This generated
    # snapshot does not need artificial cross-store shifts; validation focuses
    # on the recorded location, employee availability, and chronology.
    completed_shifts_by_store: dict[int, list[Any]] = defaultdict(list)
    cutoff_timestamp = datetime.combine(dataset_cutoff, time.max)
    for row in data["staff_shifts"].itertuples(index=False):
        if row.scheduled_start.date() < employee_hire_date[row.employee_id]:
            errors.append(
                f"staff_shifts: shift {row.shift_id} predates employee "
                f"{row.employee_id}'s hire date"
            )
        actual_start_missing = pd.isna(row.actual_start)
        actual_end_missing = pd.isna(row.actual_end)
        if row.shift_status == "completed" and (
            actual_start_missing or actual_end_missing
        ):
            errors.append(
                f"staff_shifts: completed shift {row.shift_id} lacks actual times"
            )
        if row.shift_status != "completed" and not (
            actual_start_missing and actual_end_missing
        ):
            errors.append(
                f"staff_shifts: non-completed shift {row.shift_id} has actual times"
            )
        opening_time, closing_time = store_hours[row.store_id]
        opening = datetime.combine(row.scheduled_start.date(), opening_time)
        closing = datetime.combine(row.scheduled_start.date(), closing_time)
        if not (
            opening <= row.scheduled_start < row.scheduled_end <= closing
        ):
            errors.append(
                f"staff_shifts: scheduled shift {row.shift_id} is outside store hours"
            )
        if row.shift_status == "scheduled" and row.scheduled_end <= cutoff_timestamp:
            errors.append(
                f"staff_shifts: historical shift {row.shift_id} remains scheduled"
            )
        if row.shift_status == "completed" and not (
            opening <= row.actual_start < row.actual_end <= closing
        ):
            errors.append(
                f"staff_shifts: actual shift {row.shift_id} is outside store hours"
            )
        if row.shift_status == "completed":
            completed_shifts_by_store[row.store_id].append(row)

    approved_time_off: dict[int, list[tuple[date, date]]] = defaultdict(list)
    for row in data["time_off_requests"].itertuples(index=False):
        if row.request_status == "approved":
            approved_time_off[row.employee_id].append((row.start_date, row.end_date))

    active_shifts_by_employee: dict[int, list[Any]] = defaultdict(list)
    for row in data["staff_shifts"].itertuples(index=False):
        if row.shift_status == "cancelled":
            continue
        shift_date = row.scheduled_start.date()
        if any(
            start_date <= shift_date <= end_date
            for start_date, end_date in approved_time_off.get(row.employee_id, [])
        ):
            errors.append(
                f"staff_shifts: shift {row.shift_id} conflicts with approved time off"
            )
        active_shifts_by_employee[row.employee_id].append(row)

    for employee_id, employee_shifts in active_shifts_by_employee.items():
        previous_shift = None
        for row in sorted(
            employee_shifts,
            key=lambda item: (
                item.scheduled_start,
                item.scheduled_end,
                item.shift_id,
            ),
        ):
            if (
                previous_shift is not None
                and row.scheduled_start < previous_shift.scheduled_end
            ):
                errors.append(
                    f"staff_shifts: employee {employee_id} has overlapping shifts "
                    f"{previous_shift.shift_id} and {row.shift_id}"
                )
            else:
                previous_shift = row

        previous_completed_shift = None
        for row in sorted(
            (
                item
                for item in employee_shifts
                if item.shift_status == "completed"
            ),
            key=lambda item: (item.actual_start, item.shift_id),
        ):
            if (
                previous_completed_shift is not None
                and row.actual_start < previous_completed_shift.actual_end
            ):
                errors.append(
                    f"staff_shifts: employee {employee_id} has overlapping actual "
                    f"times in shifts {previous_completed_shift.shift_id} and "
                    f"{row.shift_id}"
                )
            else:
                previous_completed_shift = row

    for row in data["sales_transactions"].itertuples(index=False):
        if not any(
            shift.actual_start <= row.sale_time <= shift.actual_end
            for shift in completed_shifts_by_store.get(row.store_id, [])
        ):
            errors.append(
                f"sales_transactions: sale {row.sale_id} has no completed "
                "shift coverage at its store and time"
            )

    # Adjustment signs.
    positive_types = {"delivery", "return"}
    negative_types = {"sale", "damage", "expired"}
    for row in data["inventory_adjustments"].itertuples(index=False):
        if row.adjustment_type in positive_types and row.quantity_delta <= 0:
            errors.append(
                f"inventory_adjustments: {row.adjustment_type} must be positive "
                f"(adjustment_id={row.adjustment_id})"
            )
        if row.adjustment_type in negative_types and row.quantity_delta >= 0:
            errors.append(
                f"inventory_adjustments: {row.adjustment_type} must be negative "
                f"(adjustment_id={row.adjustment_id})"
            )

    if (data["inventory_adjustments"]["adjustment_type"] == "sale").any():
        errors.append(
            "inventory_adjustments: sale rows must not be staged because the "
            "sales_items trigger creates them automatically"
        )

    # Rehearse the exact inventory-changing load order used below. This prevents
    # a negative-inventory trigger failure from being discovered only after the
    # database load starts.
    inventory_balance: dict[tuple[int, int], int] = defaultdict(int)
    for row in data["store_inventory"].itertuples(index=False):
        inventory_balance[(row.store_id, row.product_id)] = row.quantity_on_hand

    adjustments = data["inventory_adjustments"]
    pre_sale_adjustments = adjustments[
        adjustments["adjustment_type"].isin(["manual_count", "delivery"])
    ].sort_values(["adjustment_time", "adjustment_id"], kind="stable")
    post_sale_adjustments = adjustments[
        ~adjustments["adjustment_type"].isin(["manual_count", "delivery"])
    ].sort_values(["adjustment_time", "adjustment_id"], kind="stable")
    reported_inventory_keys: set[tuple[int, int]] = set()

    def apply_inventory_change(
        key: tuple[int, int], quantity_delta: int, source: str
    ) -> None:
        new_quantity = inventory_balance[key] + int(quantity_delta)
        if new_quantity < 0 and key not in reported_inventory_keys:
            errors.append(
                f"inventory load would become negative for store_id={key[0]}, "
                f"product_id={key[1]} during {source}"
            )
            reported_inventory_keys.add(key)
        inventory_balance[key] = new_quantity

    for row in pre_sale_adjustments.itertuples(index=False):
        apply_inventory_change(
            (row.store_id, row.product_id),
            row.quantity_delta,
            f"inventory adjustment {row.adjustment_id}",
        )

    sale_store_for_inventory = data["sales_transactions"].set_index("sale_id")[
        "store_id"
    ].to_dict()
    for row in data["sales_items"].itertuples(index=False):
        store_id = sale_store_for_inventory.get(row.sale_id)
        if store_id is None:
            continue
        apply_inventory_change(
            (store_id, row.product_id),
            -row.quantity_sold,
            f"sales item {row.sale_item_id}",
        )

    for row in post_sale_adjustments.itertuples(index=False):
        apply_inventory_change(
            (row.store_id, row.product_id),
            row.quantity_delta,
            f"inventory adjustment {row.adjustment_id}",
        )

    # Independently replay the inventory ledger by its real event timestamps.
    # A snapshot can load successfully but still imply impossible historical
    # negative stock if future deliveries were applied before earlier sales.
    chronological_balance: dict[tuple[int, int], int] = defaultdict(int)
    for row in data["store_inventory"].itertuples(index=False):
        chronological_balance[(row.store_id, row.product_id)] = row.quantity_on_hand

    chronological_events: list[
        tuple[pd.Timestamp, int, int, tuple[int, int], int, str]
    ] = []
    for row in data["inventory_adjustments"].itertuples(index=False):
        priority = 0 if row.adjustment_type in {"manual_count", "delivery"} else 2
        chronological_events.append(
            (
                row.adjustment_time,
                priority,
                row.adjustment_id,
                (row.store_id, row.product_id),
                row.quantity_delta,
                f"inventory adjustment {row.adjustment_id}",
            )
        )
    sale_time_for_inventory = data["sales_transactions"].set_index("sale_id")[
        "sale_time"
    ].to_dict()
    for row in data["sales_items"].itertuples(index=False):
        chronological_events.append(
            (
                sale_time_for_inventory[row.sale_id],
                1,
                row.sale_item_id,
                (sale_store_for_inventory[row.sale_id], row.product_id),
                -row.quantity_sold,
                f"sales item {row.sale_item_id}",
            )
        )

    chronological_negative_keys: set[tuple[int, int]] = set()
    for _, _, _, key, quantity_delta, source in sorted(chronological_events):
        chronological_balance[key] += int(quantity_delta)
        if chronological_balance[key] < 0 and key not in chronological_negative_keys:
            errors.append(
                f"inventory timeline becomes negative for store_id={key[0]}, "
                f"product_id={key[1]} during {source}"
            )
            chronological_negative_keys.add(key)

    # Purchase-order vendor must match vendor-product vendor.
    po_vendor = data["purchase_orders"].set_index("purchase_order_id")["vendor_id"].to_dict()
    vp_vendor = data["vendor_products"].set_index("vendor_product_id")["vendor_id"].to_dict()
    vp_lead_time = data["vendor_products"].set_index("vendor_product_id")[
        "lead_time_days"
    ].to_dict()
    po_items_by_order: dict[int, list[Any]] = defaultdict(list)
    for row in data["purchase_order_items"].itertuples(index=False):
        po_items_by_order[row.purchase_order_id].append(row)
        if po_vendor.get(row.purchase_order_id) != vp_vendor.get(row.vendor_product_id):
            errors.append(
                f"purchase_order_items: vendor mismatch for item "
                f"{row.purchase_order_item_id}"
            )
        if row.quantity_received + row.quantity_missing > row.quantity_ordered:
            errors.append(
                f"purchase_order_items: received + missing exceeds ordered for item "
                f"{row.purchase_order_item_id}"
            )

    deliveries_by_order: dict[int, list[Any]] = defaultdict(list)
    for row in data["deliveries"].itertuples(index=False):
        deliveries_by_order[row.purchase_order_id].append(row)

    # Purchase-order dates, fulfillment, and cutoff lifecycle.
    po_order_date = data["purchase_orders"].set_index("purchase_order_id")["order_date"].to_dict()
    for row in data["deliveries"].itertuples(index=False):
        if row.delivery_date < po_order_date[row.purchase_order_id]:
            errors.append(
                f"deliveries: delivery {row.delivery_id} occurs before its order date"
            )
        if row.delivery_date > dataset_cutoff:
            errors.append(
                f"deliveries: delivery {row.delivery_id} occurs after dataset cutoff"
            )

    for row in data["purchase_orders"].itertuples(index=False):
        order_items = po_items_by_order.get(row.purchase_order_id, [])
        if not order_items:
            errors.append(
                f"purchase_orders: order {row.purchase_order_id} has no items"
            )
            continue
        selected_max_lead_time = max(
            int(vp_lead_time[item.vendor_product_id]) for item in order_items
        )
        correct_expected_date = row.order_date + timedelta(
            days=selected_max_lead_time
        )
        if row.expected_date != correct_expected_date:
            errors.append(
                f"purchase_orders: order {row.purchase_order_id} expected date "
                "does not equal selected-item maximum lead time"
            )

        completed = row.order_status in {"received", "partially_received"}
        order_deliveries = deliveries_by_order.get(row.purchase_order_id, [])
        if completed:
            if pd.isna(row.received_date):
                errors.append(
                    f"purchase_orders: completed order {row.purchase_order_id} "
                    "lacks received date"
                )
            elif row.received_date > dataset_cutoff:
                errors.append(
                    f"purchase_orders: order {row.purchase_order_id} is received "
                    "after dataset cutoff"
                )
            if len(order_deliveries) != 1:
                errors.append(
                    f"purchase_orders: completed order {row.purchase_order_id} "
                    "must have exactly one delivery"
                )
            elif order_deliveries[0].delivery_date != row.received_date:
                errors.append(
                    f"purchase_orders: order {row.purchase_order_id} delivery "
                    "date differs from received date"
                )
            for item in order_items:
                if item.quantity_received + item.quantity_missing != item.quantity_ordered:
                    errors.append(
                        f"purchase_order_items: fulfillment does not reconcile for "
                        f"item {item.purchase_order_item_id}"
                    )
        else:
            if not pd.isna(row.received_date):
                errors.append(
                    f"purchase_orders: open/cancelled order {row.purchase_order_id} "
                    "has a received date"
                )
            if order_deliveries:
                errors.append(
                    f"purchase_orders: open/cancelled order {row.purchase_order_id} "
                    "has a delivery"
                )
            for item in order_items:
                if any(
                    value != 0
                    for value in (
                        item.quantity_received,
                        item.quantity_damaged,
                        item.quantity_missing,
                    )
                ):
                    errors.append(
                        f"purchase_order_items: unfulfilled order item "
                        f"{item.purchase_order_item_id} has receipt quantities"
                    )

        if (
            row.order_status == "submitted"
            and row.expected_date
            < dataset_cutoff - timedelta(days=OPEN_ORDER_GRACE_DAYS)
        ):
            errors.append(
                f"purchase_orders: submitted order {row.purchase_order_id} is "
                "implausibly old at the dataset cutoff"
            )

    # Sales totals must agree with sales items.
    item_df = data["sales_items"].copy()
    item_df["gross"] = item_df["quantity_sold"] * item_df["unit_retail_price"]
    item_summary = item_df.groupby("sale_id").agg(
        calculated_subtotal=("gross", "sum"),
        calculated_discount=("item_discount", "sum"),
    )
    for row in data["sales_transactions"].itertuples(index=False):
        if row.sale_status == "voided":
            continue
        if row.sale_id not in item_summary.index:
            errors.append(f"sales_transactions: sale {row.sale_id} has no sales items")
            continue
        summary = item_summary.loc[row.sale_id]
        if money(summary["calculated_subtotal"]) != money(row.subtotal_amount):
            errors.append(f"sales_transactions: subtotal mismatch for sale {row.sale_id}")
        if money(summary["calculated_discount"]) != money(row.discount_amount):
            errors.append(f"sales_transactions: discount mismatch for sale {row.sale_id}")
        expected_total = money(
            money(row.subtotal_amount) - money(row.discount_amount) + money(row.tax_amount)
        )
        if expected_total != money(row.total_amount):
            errors.append(f"sales_transactions: total mismatch for sale {row.sale_id}")

    # Payment totals.
    payment_summary = data["payments"].groupby("sale_id")["payment_amount"].sum().to_dict()
    sale_time_for_payment = data["sales_transactions"].set_index("sale_id")[
        "sale_time"
    ].to_dict()
    for row in data["payments"].itertuples(index=False):
        if row.payment_time < sale_time_for_payment[row.sale_id]:
            errors.append(
                f"payments: payment {row.payment_id} predates its sale"
            )
    for row in data["sales_transactions"].itertuples(index=False):
        if row.sale_status == "voided":
            continue
        if money(payment_summary.get(row.sale_id, 0)) != money(row.total_amount):
            errors.append(f"payments: payment total mismatch for sale {row.sale_id}")

    # Return validation.
    sale_item_qty = data["sales_items"].set_index("sale_item_id")["quantity_sold"].to_dict()
    sale_item_sale = data["sales_items"].set_index("sale_item_id")["sale_id"].to_dict()
    sale_store = data["sales_transactions"].set_index("sale_id")["store_id"].to_dict()
    returned_by_item = data["customer_returns"].groupby("sale_item_id")["quantity_returned"].sum()
    for sale_item_id, quantity_returned in returned_by_item.items():
        if quantity_returned > sale_item_qty[sale_item_id]:
            errors.append(
                f"customer_returns: returned quantity exceeds sold quantity for "
                f"sale_item_id={sale_item_id}"
            )
    for row in data["customer_returns"].itertuples(index=False):
        original_sale_id = sale_item_sale[row.sale_item_id]
        original_store = sale_store[original_sale_id]
        if row.return_time < data["sales_transactions"].set_index("sale_id").loc[
            original_sale_id, "sale_time"
        ]:
            errors.append(
                f"customer_returns: return {row.return_id} predates its original sale"
            )
        if row.return_time.date() < employee_hire_date[row.processed_by_employee_id]:
            errors.append(
                f"customer_returns: employee {row.processed_by_employee_id} processed "
                f"return {row.return_id} before being hired"
            )
        if any(
            start_date <= row.return_time.date() <= end_date
            for start_date, end_date in approved_time_off.get(
                row.processed_by_employee_id, []
            )
        ):
            errors.append(
                f"customer_returns: employee {row.processed_by_employee_id} processed "
                f"return {row.return_id} during approved time off"
            )
        opening_time, closing_time = store_hours[original_store]
        if not opening_time <= row.return_time.time() < closing_time:
            errors.append(
                f"customer_returns: return {row.return_id} occurs outside store hours"
            )
        if not any(
            shift.employee_id == row.processed_by_employee_id
            and shift.actual_start <= row.return_time <= shift.actual_end
            for shift in completed_shifts_by_store.get(original_store, [])
        ):
            errors.append(
                f"customer_returns: processor for return {row.return_id} was not "
                "working a completed shift at that store and time"
            )

    # Transaction-level refund status and amount must agree with detailed
    # customer-return records so net-sales views do not omit actual refunds.
    return_sale_id = data["customer_returns"]["sale_item_id"].map(sale_item_sale)
    refund_by_sale = (
        data["customer_returns"]
        .assign(sale_id=return_sale_id)
        .groupby("sale_id")["refund_amount"]
        .sum()
        .to_dict()
    )
    for row in data["sales_transactions"].itertuples(index=False):
        detailed_refund = money(refund_by_sale.get(row.sale_id, 0))
        if detailed_refund != money(row.refund_amount):
            errors.append(
                f"sales_transactions: refund mismatch for sale {row.sale_id}"
            )
        if row.sale_status == "completed" and detailed_refund != Decimal("0"):
            errors.append(
                f"sales_transactions: completed sale {row.sale_id} has a refund"
            )
        if row.sale_status == "partially_refunded" and not (
            Decimal("0") < detailed_refund < money(row.total_amount)
        ):
            errors.append(
                f"sales_transactions: partial-refund status mismatch for sale {row.sale_id}"
            )
        if row.sale_status == "refunded" and detailed_refund != money(row.total_amount):
            errors.append(
                f"sales_transactions: full-refund status mismatch for sale {row.sale_id}"
            )
        if row.sale_status == "voided" and detailed_refund != Decimal("0"):
            errors.append(
                f"sales_transactions: voided sale {row.sale_id} has a refund"
            )

    if errors:
        preview = "\n".join(f"- {error}" for error in errors[:50])
        raise ValueError(
            f"Business-rule validation failed with {len(errors)} issue(s):\n{preview}"
        )


def transform_and_validate(
    output_dir: Path,
    dataset_cutoff: date,
) -> dict[str, pd.DataFrame]:
    data = {
        table_name: transform_table(table_name, output_dir / f"{table_name}.csv")
        for table_name in TABLE_ORDER
    }
    validate_business_rules(data, dataset_cutoff)
    logging.info("Transformation and validation completed successfully.")
    return data


# ---------------------------------------------------------------------------
# PostgreSQL loading
# ---------------------------------------------------------------------------

def get_connection():
    """
    Create a PostgreSQL connection.

    Configure the connection through environment variables:
        PGHOST, PGPORT, PGDATABASE, PGUSER, PGPASSWORD

    Defaults are suitable for a common local PostgreSQL installation.
    """
    if psycopg2 is None:
        raise SystemExit(
            "psycopg2-binary is required for PostgreSQL loading. Install it with: "
            "pip install psycopg2-binary"
        )

    database_name = os.getenv("PGDATABASE")
    database_user = os.getenv("PGUSER")
    database_password = os.getenv("PGPASSWORD")

    if not sys.stdin.isatty() and not all(
        [database_name, database_user, database_password]
    ):
        raise SystemExit(
            "Set PGDATABASE, PGUSER, and PGPASSWORD before a non-interactive "
            "database load."
        )

    if database_name is None:
        database_name = input("Enter PostgreSQL database name: ").strip()
    if database_user is None:
        database_user = input("Enter PostgreSQL user [postgres]: ").strip() or "postgres"
    if database_password is None:
        database_password = getpass.getpass("Enter PostgreSQL password: ")
    if not database_name:
        raise SystemExit("A PostgreSQL database name is required.")

    return psycopg2.connect(
        host=os.getenv("PGHOST", "localhost"),
        port=int(os.getenv("PGPORT", "5432")),
        dbname=database_name,
        user=database_user,
        password=database_password,
    )


def ensure_database_object_compatibility(connection) -> None:
    """
    Apply idempotent trigger-function and approved view corrections.

    CREATE OR REPLACE preserves all table definitions and relationships.
    """
    with connection.cursor() as cursor:
        cursor.execute(
            f"""
            CREATE OR REPLACE FUNCTION {SCHEMA}.apply_inventory_adjustment()
            RETURNS TRIGGER AS $$
            DECLARE
                new_quantity INTEGER;
            BEGIN
                INSERT INTO {SCHEMA}.store_inventory (
                    store_id, product_id, quantity_on_hand, reorder_threshold
                )
                VALUES (NEW.store_id, NEW.product_id, 0, 0)
                ON CONFLICT (store_id, product_id) DO NOTHING;

                SELECT quantity_on_hand + NEW.quantity_delta
                INTO new_quantity
                FROM {SCHEMA}.store_inventory
                WHERE store_id = NEW.store_id AND product_id = NEW.product_id
                FOR UPDATE;

                IF new_quantity < 0 THEN
                    RAISE EXCEPTION
                        'Inventory cannot be negative for store %, product %',
                        NEW.store_id,
                        NEW.product_id;
                END IF;

                UPDATE {SCHEMA}.store_inventory
                SET quantity_on_hand = new_quantity
                WHERE store_id = NEW.store_id AND product_id = NEW.product_id;

                RETURN NEW;
            END;
            $$ LANGUAGE plpgsql;

            CREATE OR REPLACE FUNCTION {SCHEMA}.reduce_inventory_after_sale()
            RETURNS TRIGGER AS $$
            DECLARE
                sale_store_id BIGINT;
                original_sale_time TIMESTAMP;
            BEGIN
                SELECT store_id, sale_time
                INTO sale_store_id, original_sale_time
                FROM {SCHEMA}.sales_transactions
                WHERE sale_id = NEW.sale_id;

                INSERT INTO {SCHEMA}.inventory_adjustments (
                    store_id,
                    product_id,
                    adjustment_time,
                    adjustment_type,
                    quantity_delta,
                    notes
                )
                VALUES (
                    sale_store_id,
                    NEW.product_id,
                    original_sale_time,
                    'sale',
                    -NEW.quantity_sold,
                    'Automatic inventory reduction from sale item '
                        || NEW.sale_item_id
                );

                RETURN NEW;
            END;
            $$ LANGUAGE plpgsql;

            CREATE OR REPLACE FUNCTION {SCHEMA}.validate_purchase_order_vendor()
            RETURNS TRIGGER AS $$
            DECLARE
                po_vendor_id BIGINT;
                item_vendor_id BIGINT;
            BEGIN
                SELECT vendor_id
                INTO po_vendor_id
                FROM {SCHEMA}.purchase_orders
                WHERE purchase_order_id = NEW.purchase_order_id;

                SELECT vendor_id
                INTO item_vendor_id
                FROM {SCHEMA}.vendor_products
                WHERE vendor_product_id = NEW.vendor_product_id;

                IF po_vendor_id <> item_vendor_id THEN
                    RAISE EXCEPTION
                        'Purchase order item vendor must match purchase order vendor';
                END IF;

                RETURN NEW;
            END;
            $$ LANGUAGE plpgsql;

            CREATE OR REPLACE VIEW {SCHEMA}.daily_store_sales AS
            SELECT
                s.store_name,
                st.sale_time::DATE AS sale_date,
                COUNT(*) AS transaction_count,
                    SUM(
                        CASE
                            WHEN st.total_amount > 0 THEN
                                (st.subtotal_amount - st.discount_amount)
                                - ROUND(
                                    st.refund_amount
                                    * (st.subtotal_amount - st.discount_amount)
                                    / st.total_amount,
                                    2
                                )
                            ELSE 0
                        END
                    ) AS net_sales
            FROM {SCHEMA}.sales_transactions st
            JOIN {SCHEMA}.stores s ON s.store_id = st.store_id
            WHERE st.sale_status <> 'voided'
            GROUP BY s.store_name, st.sale_time::DATE;

            CREATE OR REPLACE VIEW {SCHEMA}.vendor_performance_summary AS
            WITH purchase_order_level AS (
                SELECT
                    po.purchase_order_id,
                    po.vendor_id,
                    po.order_status,
                    po.expected_date,
                    po.received_date,
                    SUM(poi.quantity_ordered) AS ordered_units,
                    SUM(poi.quantity_received) AS received_units,
                    SUM(poi.quantity_damaged) AS damaged_units,
                    SUM(poi.quantity_missing) AS missing_units
                FROM {SCHEMA}.purchase_orders po
                JOIN {SCHEMA}.purchase_order_items poi
                  ON poi.purchase_order_id = po.purchase_order_id
                GROUP BY
                    po.purchase_order_id,
                    po.vendor_id,
                    po.order_status,
                    po.expected_date,
                    po.received_date
            )
            SELECT
                v.vendor_name,
                COUNT(*) FILTER (
                    WHERE pol.order_status IN ('received', 'partially_received')
                ) AS purchase_order_count,
                    AVG(GREATEST(pol.received_date - pol.expected_date, 0)) FILTER (
                    WHERE pol.order_status IN ('received', 'partially_received')
                ) AS avg_days_late,
                COALESCE(SUM(pol.ordered_units) FILTER (
                    WHERE pol.order_status IN ('received', 'partially_received')
                ), 0)::BIGINT AS ordered_units,
                COALESCE(SUM(pol.received_units) FILTER (
                    WHERE pol.order_status IN ('received', 'partially_received')
                ), 0)::BIGINT AS received_units,
                COALESCE(SUM(pol.damaged_units) FILTER (
                    WHERE pol.order_status IN ('received', 'partially_received')
                ), 0)::BIGINT AS damaged_units,
                COALESCE(SUM(pol.missing_units) FILTER (
                    WHERE pol.order_status IN ('received', 'partially_received')
                ), 0)::BIGINT AS missing_units,
                COUNT(*) FILTER (
                    WHERE pol.order_status = 'submitted'
                ) AS open_purchase_order_count,
                COUNT(*) FILTER (
                    WHERE pol.order_status = 'cancelled'
                ) AS cancelled_purchase_order_count,
                COUNT(*) AS total_purchase_order_count,
                COALESCE(SUM(pol.ordered_units) FILTER (
                    WHERE pol.order_status = 'submitted'
                ), 0)::BIGINT AS open_ordered_units,
                COALESCE(SUM(pol.ordered_units) FILTER (
                    WHERE pol.order_status = 'cancelled'
                ), 0)::BIGINT AS cancelled_ordered_units
            FROM {SCHEMA}.vendors v
            JOIN purchase_order_level pol ON pol.vendor_id = v.vendor_id
            GROUP BY v.vendor_name;
            """
        )


def reset_target_tables(connection) -> None:
    with connection.cursor() as cursor:
        identifiers = sql.SQL(", ").join(
            sql.Identifier(SCHEMA, table_name) for table_name in TRUNCATE_TABLES
        )
        cursor.execute(
            sql.SQL("TRUNCATE TABLE {} RESTART IDENTITY CASCADE").format(identifiers)
        )
    logging.info("Existing rows were cleared inside the current load transaction.")


def require_empty_target_tables(connection) -> None:
    """Prevent --no-reset from becoming a duplicate-key append attempt."""
    nonempty_tables = []
    with connection.cursor() as cursor:
        for table_name in TABLE_ORDER:
            cursor.execute(
                sql.SQL("SELECT EXISTS (SELECT 1 FROM {}.{} LIMIT 1)").format(
                    sql.Identifier(SCHEMA),
                    sql.Identifier(table_name),
                )
            )
            if cursor.fetchone()[0]:
                nonempty_tables.append(table_name)
    if nonempty_tables:
        raise ValueError(
            "--no-reset requires empty target tables; existing rows were found in: "
            + ", ".join(nonempty_tables)
        )


def insert_dataframe(
    connection,
    table_name: str,
    df: pd.DataFrame,
    override_identity: bool = False,
) -> None:
    if df.empty:
        logging.warning("Skipping empty table %s", table_name)
        return

    columns = list(df.columns)
    values = dataframe_to_records(df)
    override_clause = sql.SQL(" OVERRIDING SYSTEM VALUE") if override_identity else sql.SQL("")

    statement = sql.SQL("INSERT INTO {}.{} ({}){} VALUES %s").format(
        sql.Identifier(SCHEMA),
        sql.Identifier(table_name),
        sql.SQL(", ").join(sql.Identifier(column) for column in columns),
        override_clause,
    )

    with connection.cursor() as cursor:
        execute_values(cursor, statement.as_string(connection), values, page_size=1000)


def load_data(connection, data: dict[str, pd.DataFrame], reset_before_load: bool) -> None:
    # Load independent and parent tables first.
    first_phase = [
        "stores",
        "job_roles",
        "employees",
        "staff_shifts",
        "time_off_requests",
        "product_categories",
        "products",
        "vendors",
        "vendor_products",
        "store_inventory",
        "purchase_orders",
        "purchase_order_items",
        "deliveries",
        "operating_expenses",
    ]

    # Inventory adjustments must be loaded in phases to guarantee enough stock
    # before sales-item triggers subtract inventory.
    adjustments = data["inventory_adjustments"].copy()
    pre_sale_adjustments = adjustments[
        adjustments["adjustment_type"].isin(["manual_count", "delivery"])
    ].sort_values(["adjustment_time", "adjustment_id"], kind="stable")
    post_sale_adjustments = adjustments[
        ~adjustments["adjustment_type"].isin(["manual_count", "delivery"])
    ].sort_values(["adjustment_time", "adjustment_id"], kind="stable")

    try:
        # Schema functions in the submitted Checkpoint 3 file used unqualified
        # table names. A fresh connection normally searches public, so keep the
        # project schema first for every trigger call in this transaction.
        with connection.cursor() as cursor:
            cursor.execute(
                sql.SQL("SET LOCAL search_path TO {}, public").format(
                    sql.Identifier(SCHEMA)
                )
            )
        ensure_database_object_compatibility(connection)

        if reset_before_load:
            reset_target_tables(connection)
        else:
            require_empty_target_tables(connection)

        for table_name in first_phase:
            insert_dataframe(
                connection,
                table_name,
                data[table_name],
                override_identity=table_name in IDENTITY_COLUMNS,
            )
            logging.info("Loaded %-24s %8d rows", table_name, len(data[table_name]))

        insert_dataframe(
            connection,
            "inventory_adjustments",
            pre_sale_adjustments,
            override_identity=True,
        )
        logging.info(
            "Loaded %-24s %8d rows",
            "inventory_adjustments (pre-sale)",
            len(pre_sale_adjustments),
        )

        # Reserve every explicit adjustment_id in the staging data before the
        # sales-item trigger starts generating its own adjustment rows.
        with connection.cursor() as cursor:
            cursor.execute(
                "SELECT setval(pg_get_serial_sequence(%s, %s), %s, true)",
                (
                    f"{SCHEMA}.inventory_adjustments",
                    "adjustment_id",
                    int(adjustments["adjustment_id"].max()),
                ),
            )

        insert_dataframe(connection, "sales_transactions", data["sales_transactions"], True)
        insert_dataframe(connection, "sales_items", data["sales_items"], True)
        insert_dataframe(connection, "payments", data["payments"], True)
        insert_dataframe(connection, "customer_returns", data["customer_returns"], True)

        logging.info("Loaded %-24s %8d rows", "sales_transactions", len(data["sales_transactions"]))
        logging.info("Loaded %-24s %8d rows", "sales_items", len(data["sales_items"]))
        logging.info("Loaded %-24s %8d rows", "payments", len(data["payments"]))
        logging.info("Loaded %-24s %8d rows", "customer_returns", len(data["customer_returns"]))

        # Existing sale-item triggers have already created "sale" adjustments.
        # Therefore, only return/damage/expired rows from the CSV are inserted here.
        insert_dataframe(
            connection,
            "inventory_adjustments",
            post_sale_adjustments,
            override_identity=True,
        )
        logging.info(
            "Loaded %-24s %8d rows",
            "inventory_adjustments (post-sale)",
            len(post_sale_adjustments),
        )

        reset_identity_sequences(connection)
        validate_loaded_database(connection, data)
        connection.commit()
    except Exception:
        connection.rollback()
        raise


def reset_identity_sequences(connection) -> None:
    """
    Move each identity sequence to the current MAX(identity_column) value after
    explicit identity values have been loaded.
    """
    with connection.cursor() as cursor:
        for table_name, identity_column in IDENTITY_COLUMNS.items():
            cursor.execute(
                """
                SELECT pg_get_serial_sequence(%s, %s)
                """,
                (f"{SCHEMA}.{table_name}", identity_column),
            )
            sequence_name = cursor.fetchone()[0]
            if not sequence_name:
                continue
            cursor.execute(
                sql.SQL("SELECT MAX({}) FROM {}.{}").format(
                    sql.Identifier(identity_column),
                    sql.Identifier(SCHEMA),
                    sql.Identifier(table_name),
                )
            )
            maximum_identity = cursor.fetchone()[0]
            if maximum_identity is None:
                # setval(..., false) makes the first generated value 1 rather
                # than incorrectly skipping to 2 for an empty identity table.
                cursor.execute(
                    "SELECT setval(%s, %s, false)",
                    (sequence_name, 1),
                )
            else:
                cursor.execute(
                    "SELECT setval(%s, %s, true)",
                    (sequence_name, int(maximum_identity)),
                )


def validate_loaded_database(
    connection,
    data: dict[str, pd.DataFrame],
) -> None:
    """Reject and roll back a load whose database state differs from staging."""
    errors: list[str] = []
    expected_counts = {
        table_name: len(data[table_name])
        for table_name in TABLE_ORDER
        if table_name not in {"store_inventory", "inventory_adjustments"}
    }

    inventory_keys = {
        (row.store_id, row.product_id)
        for row in data["store_inventory"].itertuples(index=False)
    }
    inventory_keys.update(
        (row.store_id, row.product_id)
        for row in data["inventory_adjustments"].itertuples(index=False)
    )
    sale_store = data["sales_transactions"].set_index("sale_id")["store_id"].to_dict()
    inventory_keys.update(
        (sale_store[row.sale_id], row.product_id)
        for row in data["sales_items"].itertuples(index=False)
    )
    expected_counts["store_inventory"] = len(inventory_keys)
    expected_counts["inventory_adjustments"] = (
        len(data["inventory_adjustments"]) + len(data["sales_items"])
    )

    with connection.cursor() as cursor:
        for table_name, expected_count in expected_counts.items():
            cursor.execute(
                sql.SQL("SELECT COUNT(*) FROM {}.{}").format(
                    sql.Identifier(SCHEMA),
                    sql.Identifier(table_name),
                )
            )
            actual_count = cursor.fetchone()[0]
            if actual_count != expected_count:
                errors.append(
                    f"{table_name}: expected {expected_count} rows, found {actual_count}"
                )

        cursor.execute(
            f"SELECT COUNT(*) FROM {SCHEMA}.store_inventory "
            "WHERE quantity_on_hand < 0"
        )
        if cursor.fetchone()[0] != 0:
            errors.append("store_inventory contains negative quantities")

        cursor.execute(
            f"""
            SELECT COUNT(*)
            FROM {SCHEMA}.inventory_adjustments ia
            JOIN {SCHEMA}.sales_items si
              ON ia.notes = 'Automatic inventory reduction from sale item '
                  || si.sale_item_id
            WHERE ia.adjustment_type = 'sale'
            """
        )
        if cursor.fetchone()[0] != len(data["sales_items"]):
            errors.append("sale inventory adjustments are not mapped one-to-one")

        cursor.execute(
            f"""
            SELECT COUNT(*)
            FROM {SCHEMA}.inventory_adjustments ia
            JOIN {SCHEMA}.sales_items si
              ON ia.notes = 'Automatic inventory reduction from sale item '
                  || si.sale_item_id
            JOIN {SCHEMA}.sales_transactions st ON st.sale_id = si.sale_id
            WHERE ia.adjustment_type = 'sale'
              AND ia.adjustment_time <> st.sale_time
            """
        )
        if cursor.fetchone()[0] != 0:
            errors.append("sale inventory-adjustment timestamps do not match sales")

        # Force all submitted views to execute before the transaction commits.
        for view_name in [
            "low_stock_products",
            "daily_store_sales",
            "vendor_performance_summary",
        ]:
            cursor.execute(
                sql.SQL("SELECT COUNT(*) FROM {}.{}").format(
                    sql.Identifier(SCHEMA),
                    sql.Identifier(view_name),
                )
            )
            cursor.fetchone()

        cursor.execute(
            f"SELECT COALESCE(SUM(transaction_count), 0), "
            f"COALESCE(SUM(net_sales), 0) FROM {SCHEMA}.daily_store_sales"
        )
        daily_count, daily_net_sales = cursor.fetchone()
        cursor.execute(
            f"""
            SELECT
                COUNT(*),
                COALESCE(SUM(
                    CASE
                        WHEN total_amount > 0 THEN
                            (subtotal_amount - discount_amount)
                            - ROUND(
                                refund_amount
                                * (subtotal_amount - discount_amount)
                                / total_amount,
                                2
                            )
                        ELSE 0
                    END
                ), 0)
            FROM {SCHEMA}.sales_transactions
            WHERE sale_status <> 'voided'
            """
        )
        base_count, base_net_sales = cursor.fetchone()
        if daily_count != base_count:
            errors.append(
                "daily_store_sales transaction_count excludes or double-counts "
                "non-voided transactions"
            )
        if money(daily_net_sales) != money(base_net_sales):
            errors.append("daily_store_sales net_sales differs from base transactions")

        cursor.execute(
            f"""
            SELECT
                COALESCE(SUM(purchase_order_count), 0),
                COALESCE(SUM(open_purchase_order_count), 0),
                COALESCE(SUM(cancelled_purchase_order_count), 0),
                COALESCE(SUM(total_purchase_order_count), 0)
            FROM {SCHEMA}.vendor_performance_summary
            """
        )
        view_completed, view_open, view_cancelled, view_total = cursor.fetchone()
        cursor.execute(
            f"""
            SELECT
                COUNT(*) FILTER (
                    WHERE order_status IN ('received', 'partially_received')
                ),
                COUNT(*) FILTER (WHERE order_status = 'submitted'),
                COUNT(*) FILTER (WHERE order_status = 'cancelled'),
                COUNT(*)
            FROM {SCHEMA}.purchase_orders
            """
        )
        base_completed, base_open, base_cancelled, base_total = cursor.fetchone()
        if (
            view_completed,
            view_open,
            view_cancelled,
            view_total,
        ) != (
            base_completed,
            base_open,
            base_cancelled,
            base_total,
        ):
            errors.append(
                "vendor_performance_summary status populations do not reconcile"
            )

    if errors:
        raise ValueError("Post-load validation failed: " + "; ".join(errors))
    logging.info("Post-load database validation completed successfully.")


def database_validation_report(connection) -> list[tuple[str, Any]]:
    checks = {
        "stores": f"SELECT COUNT(*) FROM {SCHEMA}.stores",
        "employees": f"SELECT COUNT(*) FROM {SCHEMA}.employees",
        "products": f"SELECT COUNT(*) FROM {SCHEMA}.products",
        "store_inventory": f"SELECT COUNT(*) FROM {SCHEMA}.store_inventory",
        "sales_transactions": f"SELECT COUNT(*) FROM {SCHEMA}.sales_transactions",
        "sales_items": f"SELECT COUNT(*) FROM {SCHEMA}.sales_items",
        "inventory_adjustments": f"SELECT COUNT(*) FROM {SCHEMA}.inventory_adjustments",
        "sale_adjustments": (
            f"SELECT COUNT(*) FROM {SCHEMA}.inventory_adjustments "
            "WHERE adjustment_type = 'sale'"
        ),
        "sale_adjustment_time_mismatches": (
            f"SELECT COUNT(*) FROM {SCHEMA}.inventory_adjustments ia "
            f"JOIN {SCHEMA}.sales_items si "
            "ON ia.notes = 'Automatic inventory reduction from sale item ' "
            "|| si.sale_item_id "
            f"JOIN {SCHEMA}.sales_transactions st ON st.sale_id = si.sale_id "
            "WHERE ia.adjustment_type = 'sale' "
            "AND ia.adjustment_time <> st.sale_time"
        ),
        "negative_inventory": (
            f"SELECT COUNT(*) FROM {SCHEMA}.store_inventory "
            "WHERE quantity_on_hand < 0"
        ),
        "low_stock_rows": f"SELECT COUNT(*) FROM {SCHEMA}.low_stock_products",
        "daily_nonvoided_transactions": (
            f"SELECT COALESCE(SUM(transaction_count), 0) "
            f"FROM {SCHEMA}.daily_store_sales"
        ),
        "base_nonvoided_transactions": (
            f"SELECT COUNT(*) FROM {SCHEMA}.sales_transactions "
            "WHERE sale_status <> 'voided'"
        ),
        "vendor_completed_orders": (
            f"SELECT COALESCE(SUM(purchase_order_count), 0) "
            f"FROM {SCHEMA}.vendor_performance_summary"
        ),
        "vendor_open_orders": (
            f"SELECT COALESCE(SUM(open_purchase_order_count), 0) "
            f"FROM {SCHEMA}.vendor_performance_summary"
        ),
        "vendor_cancelled_orders": (
            f"SELECT COALESCE(SUM(cancelled_purchase_order_count), 0) "
            f"FROM {SCHEMA}.vendor_performance_summary"
        ),
    }

    results = []
    with connection.cursor() as cursor:
        for label, query in checks.items():
            cursor.execute(query)
            results.append((label, cursor.fetchone()[0]))
    return results


# ---------------------------------------------------------------------------
# Command-line interface
# ---------------------------------------------------------------------------

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Generate, transform, validate, and load ABC Foodmart data."
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path("abc_foodmart_staging"),
        help="Directory for generated staging CSV files.",
    )
    parser.add_argument(
        "--start-date",
        type=date.fromisoformat,
        default=DEFAULT_START_DATE,
        help="Synthetic-data start date in YYYY-MM-DD format.",
    )
    parser.add_argument(
        "--end-date",
        type=date.fromisoformat,
        default=DEFAULT_END_DATE,
        help="Synthetic-data end date in YYYY-MM-DD format.",
    )
    parser.add_argument(
        "--sales-count",
        type=int,
        default=DEFAULT_SALES_COUNT,
        help="Number of synthetic sales transactions to generate.",
    )
    parser.add_argument(
        "--employee-count",
        type=int,
        default=DEFAULT_EMPLOYEE_COUNT,
        help="Number of synthetic employees to generate.",
    )
    parser.add_argument(
        "--no-reset",
        action="store_true",
        help="Do not truncate existing target tables before loading.",
    )
    parser.add_argument(
        "--generate-only",
        action="store_true",
        help="Generate and validate CSV files without loading PostgreSQL.",
    )
    parser.add_argument(
        "--load-only",
        action="store_true",
        help="Load existing CSV files without regenerating them.",
    )
    return parser.parse_args()


def main() -> int:
    args = parse_args()
    if args.generate_only and args.load_only:
        raise SystemExit("--generate-only and --load-only cannot be used together.")
    if args.end_date < args.start_date:
        raise SystemExit("--end-date cannot be earlier than --start-date.")
    if args.sales_count <= 0 or args.employee_count < 3:
        raise SystemExit("sales-count must be positive and employee-count must be at least 3.")

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
    )

    config = PipelineConfig(
        output_dir=args.output_dir,
        start_date=args.start_date,
        end_date=args.end_date,
        sales_count=args.sales_count,
        employee_count=args.employee_count,
        reset_before_load=not args.no_reset,
        generate_only=args.generate_only,
        load_only=args.load_only,
    )

    if not config.load_only:
        logging.info("Generating synthetic data with random seed %s", RANDOM_SEED)
        generate_all_data(config)

    data = transform_and_validate(config.output_dir, config.end_date)

    if config.generate_only:
        logging.info("Generation-only mode completed. No database changes were made.")
        return 0

    connection = get_connection()
    try:
        load_data(connection, data, config.reset_before_load)
        logging.info("Database load completed successfully.")
        for label, result in database_validation_report(connection):
            logging.info("Validation %-24s %s", label, result)
    finally:
        connection.close()

    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
"""
Generate realistic synthetic staffing and time-off data for ABC Foodmart.

The script is deterministic for a given seed. It writes preview CSV files and a
validation report. With --apply, it replaces only staff_shifts and
time_off_requests, inside one PostgreSQL transaction.

"""

In [ ]:
DATA_START = date(2025, 11, 1)
DATA_END = date(2026, 7, 31)


@dataclass(frozen=True)
class Employee:
    employee_id: int
    store_id: int
    role_name: str
    hire_date: date
    employment_status: str


@dataclass(frozen=True)
class Store:
    store_id: int
    store_name: str
    opening_time: time
    closing_time: time


def connect():
    return psycopg2.connect(
        host=os.environ.get("PGHOST", "localhost"),
        port=os.environ.get("PGPORT", "5432"),
        dbname=os.environ.get("PGDATABASE", "abc_foodmart_final"),
        user=os.environ.get("PGUSER", "postgres"),
        password=os.environ["PGPASSWORD"],
    )


def load_reference_data(connection) -> tuple[list[Employee], list[Store]]:
    with connection.cursor(cursor_factory=RealDictCursor) as cursor:
        cursor.execute(
            """
            SELECT e.employee_id, e.store_id, jr.role_name, e.hire_date,
                   e.employment_status
            FROM abc_foodmart.employees e
            JOIN abc_foodmart.job_roles jr ON jr.role_id = e.role_id
            ORDER BY e.employee_id
            """
        )
        employees = [Employee(**row) for row in cursor.fetchall()]

        cursor.execute(
            """
            SELECT store_id, store_name, opening_time, closing_time
            FROM abc_foodmart.stores
            WHERE operational_status = 'active'
            ORDER BY store_id
            """
        )
        stores = [Store(**row) for row in cursor.fetchall()]
    return employees, stores


def daterange(start: date, end: date) -> Iterable[date]:
    current = start
    while current <= end:
        yield current
        current += timedelta(days=1)


def random_date(rng: random.Random, start: date, end: date) -> date:
    return start + timedelta(days=rng.randint(0, (end - start).days))


def generate_time_off(
    rng: random.Random, employees: list[Employee]
) -> tuple[list[tuple], dict[int, set[date]]]:
    reasons = [
        "Personal leave",
        "Medical appointment",
        "Family responsibility",
        "Vacation",
        "School commitment",
    ]
    rows: list[tuple] = []
    approved_days: dict[int, set[date]] = defaultdict(set)
    request_id = 1

    for employee in employees:
        if employee.employment_status != "active":
            continue
        earliest_start = max(DATA_START + timedelta(days=14), employee.hire_date)
        if earliest_start > DATA_END:
            continue

        request_count = rng.choices([0, 1, 2], weights=[0.15, 0.65, 0.20], k=1)[0]
        used_ranges: list[tuple[date, date]] = []
        for _ in range(request_count):
            for _attempt in range(100):
                start = random_date(rng, earliest_start, DATA_END)
                duration = rng.choices([1, 2, 3, 4, 5], weights=[45, 30, 15, 7, 3], k=1)[0]
                end = min(start + timedelta(days=duration - 1), DATA_END)
                if all(end < old_start or start > old_end for old_start, old_end in used_ranges):
                    break
            else:
                continue

            status = rng.choices(
                ["approved", "denied", "cancelled"], weights=[62, 24, 14], k=1
            )[0]
            request_date = max(
                employee.hire_date,
                DATA_START,
                start - timedelta(days=rng.randint(7, 30)),
            )
            review_date = None
            if status in {"approved", "denied"}:
                review_date = min(start, request_date + timedelta(days=rng.randint(1, 5)))

            rows.append(
                (
                    request_id,
                    employee.employee_id,
                    request_date,
                    start,
                    end,
                    rng.choice(reasons),
                    status,
                    review_date,
                )
            )
            used_ranges.append((start, end))
            if status == "approved":
                approved_days[employee.employee_id].update(daterange(start, end))
            request_id += 1

    return rows, approved_days


def role_group(role_name: str) -> str:
    if role_name in {"Store Manager", "Assistant Manager"}:
        return "management"
    if role_name in {"Cashier", "Customer Service Associate"}:
        return "service"
    if role_name in {"Stock Clerk", "Inventory Specialist"}:
        return "inventory"
    raise ValueError(f"Unsupported role: {role_name}")


def weekly_target(role_name: str) -> float:
    return {
        "Store Manager": 36.0,
        "Assistant Manager": 28.0,
        "Cashier": 16.0,
        "Customer Service Associate": 18.0,
        "Stock Clerk": 16.0,
        "Inventory Specialist": 22.0,
    }[role_name]


def weekly_cap(role_name: str) -> float:
    return 40.0 if role_name in {"Store Manager", "Assistant Manager"} else 32.0


def combine(day: date, clock: time) -> datetime:
    return datetime.combine(day, clock)


def choose_employee(
    rng: random.Random,
    candidates: list[Employee],
    day: date,
    duration_hours: float,
    approved_days: dict[int, set[date]],
    assigned_today: set[int],
    weekly_hours: dict[tuple[int, date], float],
    total_shifts: Counter,
) -> Employee | None:
    week_start = day - timedelta(days=day.weekday())
    eligible = [
        employee
        for employee in candidates
        if employee.hire_date <= day
        and day not in approved_days.get(employee.employee_id, set())
        and employee.employee_id not in assigned_today
        and weekly_hours[(employee.employee_id, week_start)] + duration_hours
        <= weekly_cap(employee.role_name)
    ]
    if not eligible:
        return None

    rng.shuffle(eligible)
    eligible.sort(
        key=lambda employee: (
            weekly_hours[(employee.employee_id, week_start)]
            / weekly_target(employee.role_name),
            total_shifts[employee.employee_id],
        )
    )
    return eligible[0]


def generate_shifts(
    rng: random.Random,
    employees: list[Employee],
    stores: list[Store],
    approved_days: dict[int, set[date]],
) -> tuple[list[tuple], list[str]]:
    active = [employee for employee in employees if employee.employment_status == "active"]
    employee_pools: dict[tuple[int, str], list[Employee]] = defaultdict(list)
    for employee in active:
        employee_pools[(employee.store_id, role_group(employee.role_name))].append(employee)

    employee_miss_rate = {
        employee.employee_id: rng.uniform(0.008, 0.025) for employee in active
    }
    employee_late_rate = {
        employee.employee_id: rng.uniform(0.025, 0.075) for employee in active
    }
    employee_start_bias = {
        employee.employee_id: rng.uniform(-1.5, 1.5) for employee in active
    }

    weekly_hours: dict[tuple[int, date], float] = defaultdict(float)
    total_shifts: Counter = Counter()
    rows: list[tuple] = []
    missing_slots: list[str] = []
    shift_id = 1

    for day in daterange(DATA_START, DATA_END):
        for store in stores:
            opening = combine(day, store.opening_time)
            closing = combine(day, store.closing_time)

            slots: list[tuple[str, datetime, datetime]] = [
                # The schema has no shift-lead role. One manager/assistant is
                # scheduled for the central eight-hour management shift; the
                # opening and closing service teams cover the remaining hours.
                (
                    "management",
                    opening + timedelta(hours=1),
                    min(opening + timedelta(hours=9), closing),
                ),
                ("service", opening, min(opening + timedelta(hours=6), closing)),
                ("service", max(closing - timedelta(hours=6), opening), closing),
                (
                    "inventory",
                    opening + timedelta(hours=2),
                    min(opening + timedelta(hours=8), closing),
                ),
            ]
            if day.weekday() in {4, 5, 6}:  # Friday through Sunday support peak
                support_start = opening + timedelta(hours=4)
                slots.append(
                    ("service", support_start, min(support_start + timedelta(hours=5), closing))
                )

            assigned_today: set[int] = set()
            for group_name, scheduled_start, scheduled_end in slots:
                duration = (scheduled_end - scheduled_start).total_seconds() / 3600.0
                candidate = choose_employee(
                    rng,
                    employee_pools[(store.store_id, group_name)],
                    day,
                    duration,
                    approved_days,
                    assigned_today,
                    weekly_hours,
                    total_shifts,
                )
                if candidate is None:
                    missing_slots.append(f"{day}|{store.store_id}|{group_name}")
                    continue

                assigned_today.add(candidate.employee_id)
                week_start = day - timedelta(days=day.weekday())
                weekly_hours[(candidate.employee_id, week_start)] += duration
                total_shifts[candidate.employee_id] += 1

                draw = rng.random()
                if draw < 0.04:
                    shift_status = "cancelled"
                    actual_start = None
                    actual_end = None
                elif draw < 0.04 + employee_miss_rate[candidate.employee_id]:
                    shift_status = "missed"
                    actual_start = None
                    actual_end = None
                else:
                    shift_status = "completed"
                    if rng.random() < employee_late_rate[candidate.employee_id]:
                        start_variance = rng.randint(6, 18)
                    else:
                        start_variance = round(
                            rng.gauss(employee_start_bias[candidate.employee_id], 2.7)
                        )
                        start_variance = max(-10, min(5, start_variance))
                    end_variance = max(-12, min(15, round(rng.gauss(0, 5.5))))
                    actual_start = scheduled_start + timedelta(minutes=start_variance)
                    actual_end = scheduled_end + timedelta(minutes=end_variance)
                    if actual_end <= actual_start + timedelta(hours=3):
                        actual_end = actual_start + timedelta(hours=3, minutes=30)

                rows.append(
                    (
                        shift_id,
                        candidate.employee_id,
                        store.store_id,
                        scheduled_start,
                        scheduled_end,
                        actual_start,
                        actual_end,
                        shift_status,
                    )
                )
                shift_id += 1

    return rows, missing_slots


def validate(
    employees: list[Employee],
    stores: list[Store],
    shifts: list[tuple],
    time_off: list[tuple],
) -> dict:
    employee_by_id = {employee.employee_id: employee for employee in employees}
    store_by_id = {store.store_id: store for store in stores}
    errors: list[str] = []

    shift_ids = [row[0] for row in shifts]
    if len(shift_ids) != len(set(shift_ids)):
        errors.append("Duplicate shift IDs")

    requests_by_employee: dict[int, list[tuple]] = defaultdict(list)
    for request in time_off:
        requests_by_employee[request[1]].append(request)
        if request[4] < request[3]:
            errors.append(f"Invalid time-off date order: {request[0]}")
        if request[6] in {"approved", "denied"} and request[7] is None:
            errors.append(f"Missing review date: {request[0]}")

    by_employee_day: dict[tuple[int, date], list[tuple]] = defaultdict(list)
    by_employee_week_hours: dict[tuple[int, date], float] = defaultdict(float)
    for shift in shifts:
        (
            shift_id,
            employee_id,
            store_id,
            scheduled_start,
            scheduled_end,
            actual_start,
            actual_end,
            status,
        ) = shift
        employee = employee_by_id[employee_id]
        store = store_by_id[store_id]
        day = scheduled_start.date()
        by_employee_day[(employee_id, day)].append(shift)
        week = day - timedelta(days=day.weekday())
        by_employee_week_hours[(employee_id, week)] += (
            scheduled_end - scheduled_start
        ).total_seconds() / 3600.0

        if employee.employment_status != "active":
            errors.append(f"Non-active employee scheduled: {shift_id}")
        if employee.store_id != store_id:
            errors.append(f"Employee/store mismatch: {shift_id}")
        if day < employee.hire_date:
            errors.append(f"Shift before hire date: {shift_id}")
        if scheduled_end <= scheduled_start:
            errors.append(f"Invalid scheduled duration: {shift_id}")
        if scheduled_start.time() < store.opening_time or scheduled_end.time() > store.closing_time:
            errors.append(f"Shift outside store hours: {shift_id}")
        if status == "completed":
            if actual_start is None or actual_end is None or actual_end <= actual_start:
                errors.append(f"Invalid completed actual time: {shift_id}")
        elif actual_start is not None or actual_end is not None:
            errors.append(f"Non-completed shift has actual time: {shift_id}")

        for request in requests_by_employee[employee_id]:
            if request[6] == "approved" and request[3] <= day <= request[4]:
                errors.append(f"Shift conflicts with approved time off: {shift_id}")

    daily_multi_shift_count = sum(len(rows) > 1 for rows in by_employee_day.values())
    if daily_multi_shift_count:
        errors.append(f"Employee-days with multiple shifts: {daily_multi_shift_count}")

    weekly_over_40_count = sum(hours > 40.0 for hours in by_employee_week_hours.values())
    if weekly_over_40_count:
        errors.append(f"Employee-weeks over 40 hours: {weekly_over_40_count}")

    status_counts = Counter(row[7] for row in shifts)
    completed = status_counts["completed"]
    missed = status_counts["missed"]
    late_over_five = sum(
        row[7] == "completed"
        and (row[5] - row[3]).total_seconds() / 60.0 > 5
        for row in shifts
    )
    scheduled_hours = sum(
        (row[4] - row[3]).total_seconds() / 3600.0 for row in shifts
    )
    actual_hours = sum(
        (row[6] - row[5]).total_seconds() / 3600.0
        for row in shifts
        if row[7] == "completed"
    )
    active_employee_ids = {
        employee.employee_id
        for employee in employees
        if employee.employment_status == "active"
    }
    scheduled_employee_ids = {row[1] for row in shifts}
    monthly_counts: Counter = Counter(
        (row[1], row[3].strftime("%Y-%m")) for row in shifts
    )

    report = {
        "valid": not errors,
        "errors": errors,
        "date_start": DATA_START.isoformat(),
        "date_end": DATA_END.isoformat(),
        "active_employees": len(active_employee_ids),
        "active_employees_without_shifts": len(active_employee_ids - scheduled_employee_ids),
        "shift_rows": len(shifts),
        "time_off_rows": len(time_off),
        "shift_status_counts": dict(status_counts),
        "attendance_rate_pct": round(100.0 * completed / (completed + missed), 2),
        "late_over_5_count": late_over_five,
        "late_over_5_pct": round(100.0 * late_over_five / completed, 2),
        "scheduled_hours": round(scheduled_hours, 1),
        "actual_hours": round(actual_hours, 1),
        "max_weekly_scheduled_hours": round(max(by_employee_week_hours.values()), 1),
        "employee_days_with_multiple_shifts": daily_multi_shift_count,
        "employee_weeks_over_40_hours": weekly_over_40_count,
        "max_monthly_shifts_per_employee": max(monthly_counts.values()),
        "approved_time_off_requests": sum(row[6] == "approved" for row in time_off),
        "denied_time_off_requests": sum(row[6] == "denied" for row in time_off),
        "cancelled_time_off_requests": sum(row[6] == "cancelled" for row in time_off),
    }
    return report


def write_csv(path: Path, headers: list[str], rows: list[tuple]) -> None:
    with path.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(headers)
        writer.writerows(rows)


def apply_to_database(connection, shifts: list[tuple], time_off: list[tuple]) -> None:
    with connection:
        with connection.cursor() as cursor:
            cursor.execute(
                "CREATE TEMP TABLE staged_staff_shifts "
                "(LIKE abc_foodmart.staff_shifts INCLUDING DEFAULTS INCLUDING CONSTRAINTS) ON COMMIT DROP"
            )
            cursor.execute(
                "CREATE TEMP TABLE staged_time_off_requests "
                "(LIKE abc_foodmart.time_off_requests INCLUDING DEFAULTS INCLUDING CONSTRAINTS) ON COMMIT DROP"
            )
            execute_values(
                cursor,
                """
                INSERT INTO staged_staff_shifts
                (shift_id,employee_id,store_id,scheduled_start,scheduled_end,
                 actual_start,actual_end,shift_status)
                OVERRIDING SYSTEM VALUE VALUES %s
                """,
                shifts,
                page_size=1000,
            )
            execute_values(
                cursor,
                """
                INSERT INTO staged_time_off_requests
                (request_id,employee_id,request_date,start_date,end_date,reason,
                 request_status,review_date)
                OVERRIDING SYSTEM VALUE VALUES %s
                """,
                time_off,
                page_size=1000,
            )

            cursor.execute(
                "TRUNCATE TABLE abc_foodmart.staff_shifts, abc_foodmart.time_off_requests"
            )
            cursor.execute(
                "INSERT INTO abc_foodmart.staff_shifts OVERRIDING SYSTEM VALUE "
                "SELECT * FROM staged_staff_shifts"
            )
            cursor.execute(
                "INSERT INTO abc_foodmart.time_off_requests OVERRIDING SYSTEM VALUE "
                "SELECT * FROM staged_time_off_requests"
            )


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--seed", type=int, default=20260812)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--apply", action="store_true")
    args = parser.parse_args()

    args.output_dir.mkdir(parents=True, exist_ok=True)
    rng = random.Random(args.seed)
    connection = connect()
    try:
        employees, stores = load_reference_data(connection)
        time_off, approved_days = generate_time_off(rng, employees)
        shifts, missing_slots = generate_shifts(
            rng, employees, stores, approved_days
        )
        report = validate(employees, stores, shifts, time_off)
        report["seed"] = args.seed
        report["unfilled_coverage_slots"] = len(missing_slots)

        write_csv(
            args.output_dir / "staff_shifts_realistic.csv",
            [
                "shift_id",
                "employee_id",
                "store_id",
                "scheduled_start",
                "scheduled_end",
                "actual_start",
                "actual_end",
                "shift_status",
            ],
            shifts,
        )
        write_csv(
            args.output_dir / "time_off_requests_realistic.csv",
            [
                "request_id",
                "employee_id",
                "request_date",
                "start_date",
                "end_date",
                "reason",
                "request_status",
                "review_date",
            ],
            time_off,
        )
        (args.output_dir / "validation_report.json").write_text(
            json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8"
        )

        if not report["valid"]:
            raise RuntimeError("Generated data failed validation; database was not changed.")
        if missing_slots:
            raise RuntimeError(
                f"Generated schedule has {len(missing_slots)} unfilled coverage slots; "
                "database was not changed."
            )
        if args.apply:
            apply_to_database(connection, shifts, time_off)
            report["applied"] = True
            (args.output_dir / "validation_report.json").write_text(
                json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8"
            )
        else:
            report["applied"] = False

        print(json.dumps(report, indent=2, ensure_ascii=False))
    finally:
        connection.close()


if __name__ == "__main__":
    main()
